# 低精度矩阵乘法优化

矩阵乘法（GEMM），即$C=A\times B$，是科学计算、数值模拟和人工智能计算中的典型基础运算，其性能与数据精度、存储开销、数据搬运量、分块方式和并行效率密切相关。在高性能计算的算子开发中，低精度计算是优化算子性能的重要手段之一，通过减少数据存储量和数据搬运量，从而提高处理器的计算吞吐率。

本实验围绕稠密矩阵乘法展开，在Ascend C环境中采用静态Tensor编程方式实现。首先在CPU上实现串行FP64 GEMM，作为正确性与性能基线；随后基于Ascend C，在昇腾NPU的AI Vector核上分别实现FP32和FP16 GEMM，并比较不同精度下的数值误差与执行性能。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建实验目录、加载CANN环境并写入工程公共文件；
3. 问题分析：分析GEMM公式、低精度数据、分块方法、数据流和实验参数；
4. 核函数开发：实现FP32、FP16 GEMM核函数；
5. 结果验证与性能分析：完成数据准备、工程构建、算子运行、结果验证和性能分析；
6. 实验总结：归纳低精度转换、Vector上实现矩阵计算、同步管理和性能分析。


---
## 1. 实验概述

本实验以$C=A\times B$为问题背景，在Ascend C环境下研究数据精度对稠密矩阵乘法的存储、计算和误差特征的影响。输入矩阵$A$的形状为$M\times K$，输入矩阵$B$的形状为$K\times N$，输出矩阵$C$的形状为$M\times N$。

实验首先在Host侧生成确定性的FP64输入，并分别显式转换为FP32和FP16；随后以CPU FP64串行GEMM生成参考结果，在昇腾NPU的AI Vector核上运行采用静态Tensor编程方式实现的FP32和FP16核函数；最后从最大绝对误差、最大相对误差、端到端时间和加速比分析实验结果。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解低精度的实现。能够理解FP64、FP32和FP16浮点数据在存储空间、有效位数和数值范围方面的差异；认识低精度对数据存储量、数据搬运量、计算吞吐率和数值误差的影响。
2. 掌握GEMM的实现。能够按照矩阵规模和目标精度设置数据分块，在昇腾NPU的Vector核上实现FP32和FP16矩阵乘法；理解行方向任务划分、列方向分块和多核并行之间的关系。
3. 具备正确性验证和性能分析能力。能够以CPU FP64 GEMM作为基线，通过最大绝对误差和最大相对误差，分别验证FP32及FP16数据精度下GEMM结果的正确性；能够记录总执行时间并计算加速比，分析低精度对GEMM计算误差和执行性能的影响。


### 1.2 前置知识

本实验要求提前具备以下基础：

1. GEMM基础：理解GEMM的数学计算过程，即输入矩阵A的形状为M×K、输入矩阵B的形状为K×N，输出矩阵C的形状为M×N，其中每个输出元素由左矩阵对应行与右矩阵对应列进行乘加计算得到。认识矩阵维度与计算量、数据访问量之间的关系，理解M、N方向决定输出矩阵规模，K方向为乘加计算的归约维度。
2. 低精度计算基础：理解FP64、FP32和FP16浮点数据在存储空间、有效位数和数值范围方面的差异。认识低精度计算可以减少输入矩阵的存储量和数据搬运量，但精度转换和乘加计算会产生一定的数值误差。
3. 数据分块基础：理解将大规模矩阵划分为若干较小数据块的方法。认识数据块的行列位置、块大小和数据存储范围之间的关系，并理解通过数据分块将GEMM任务分配给多个AI Core的并行计算思想。
4. Ascend C开发基础：Host侧负责输入数据准备、数据类型转换、Device侧内存申请、Tiling参数生成、Kernel启动和结果校验；Device侧执行Kernel核函数，完成矩阵数据分块、GEMM和结果写回。实验前应熟悉Ascend C工程编译、运行和调试方法。
5. 静态Tensor编程基础：能够根据数据分块规模划分片上存储空间，正确管理数据搬入、计算和结果写回之间的依赖关系；并理解数据搬运过程与实现。

### 1.3 实验要点

实验中应重点关注以下内容：

1. 数据组织：按照连续内存方式保存FP64、FP32和FP16矩阵数据，确保Host侧和Device侧对矩阵维度、数据类型、存储顺序和分块方式保持一致。
2. 低精度转换：在Host侧生成FP64输入矩阵，并将其分别转换为FP32和FP16数据，理解浮点数据转换方法，以及降低输入精度对存储空间、数据搬运量和数值误差的影响。
3. GEMM实现：在Host侧完成FP64输入数据生成、低精度转换及实验参数准备；在Device侧的AI Vector核上分别完成FP32和FP16 GEMM，并通过行方向任务划分和数据分块实现多核并行计算。
4. 结果验证与性能分析：使用CPU FP64计算结果验证FP32和FP16算子的正确性；记录两种算子的执行时间并计算加速比，分析不同低精度计算对GEMM性能的影响。


---
## 2. 环境准备

### 2.1 创建实验目录并加载CANN环境

本小节创建实验所需目录，并加载Ascend CANN环境变量。

目录划分如下：

- `src/04.01_extra_low_precision_gemm/include`：保存公共数据结构、误差统计、CPU接口和Vector配置；
- `src/04.01_extra_low_precision_gemm/src`：保存输入生成、精度转换、CPU FP64参考实现和CPU演示程序；
- `src/04.01_extra_low_precision_gemm/ascend_ops/op_kernel`：保存采用静态Tensor编程方式实现的FP32和FP16 GEMM核函数；
- `src/04.01_extra_low_precision_gemm/ascend_ops/host_launch`：保存ACL运行时封装、Tiling配置和Host侧启动程序；
- `src/04.01_extra_low_precision_gemm/scripts`：保存CPU与NPU构建运行脚本；
- `src/04.01_extra_low_precision_gemm/results`：保存运行日志与JSON实验结果。

若当前环境没有安装CANN或没有NPU，仍可运行源码生成、参数分析和CPU基线单元；NPU构建与运行单元会给出跳过提示。


In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess

WORK_DIR = Path("src/04.01_extra_low_precision_gemm").resolve()
SRC_DIR = WORK_DIR / "src"

# 创建完整实验目录
for directory in [
    WORK_DIR / "results",
    WORK_DIR / "scripts",
    WORK_DIR / "include",
    SRC_DIR,
    WORK_DIR / "ascend_ops" / "op_kernel",
    WORK_DIR / "ascend_ops" / "host_launch",
]:
    directory.mkdir(parents=True, exist_ok=True)

# 自动查找并加载CANN环境
arch = os.uname().machine
candidate_paths = []
for name in ["ASCEND_INSTALL_PATH", "ASCEND_TOOLKIT_HOME"]:
    value = os.environ.get(name)
    if value:
        candidate_paths.append(Path(value))

ascend_home = os.environ.get("ASCEND_HOME_PATH")
if ascend_home:
    candidate_paths.extend([Path(ascend_home), Path(ascend_home) / f"{arch}-linux"])

candidate_paths.extend(
    sorted(Path("/opt/conda/Ascend").glob(f"cann-*/{arch}-linux"), reverse=True)
)
candidate_paths.append(Path("/usr/local/Ascend/ascend-toolkit/latest"))
candidate_paths.extend(
    sorted(Path("/usr/local/Ascend/ascend-toolkit").glob(f"*/{arch}-linux"), reverse=True)
)

set_env = None
for item in candidate_paths:
    if (item / "set_env.sh").exists():
        set_env = item / "set_env.sh"
        break

if set_env is not None:
    command = f"source {shlex.quote(str(set_env))} && env"
    loaded_env = subprocess.check_output(["bash", "-lc", command], text=True)
    for line in loaded_env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(set_env.parent)
    print("Ascend environment loaded from:", set_env)
else:
    print("Ascend set_env.sh was not found; NPU build/run will be skipped.")

CANN_AVAILABLE = set_env is not None
NPU_AVAILABLE = shutil.which("npu-smi") is not None
CMAKE_AVAILABLE = shutil.which("cmake") is not None

print("Experiment directory:", WORK_DIR)
print("CANN / NPU / CMake:", CANN_AVAILABLE, NPU_AVAILABLE, CMAKE_AVAILABLE)


### 2.2 写入工程公共文件

本节写入公共头文件、Host侧输入与精度转换代码、CPU FP64参考实现、ACL运行时封装和NPU Host侧启动代码。Device侧核函数将在第4节写入，CMake文件和运行脚本将在第5节写入。

顺序执行到第5节后，`src/04.01_extra_low_precision_gemm`目录将形成一套可独立编译执行的Ascend C工程。代码来自与本实验手册配套的`low_precision_GEMM`实现，Notebook与工程保持同源。


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/include/gemm_common.h
#ifndef LOW_PRECISION_GEMM_COMMON_H
#define LOW_PRECISION_GEMM_COMMON_H

#include <cstddef>
#include <cstdint>
#include <string>
#include <vector>

namespace lowp {

struct MatrixShape {
    std::size_t m;
    std::size_t n;
    std::size_t k;
};

struct ErrorMetrics {
    double maxAbsoluteError = 0.0;
    double maxRelativeError = 0.0;
    double relativeDenominatorFloor = 1.0e-12;
    bool allFinite = true;
};

std::size_t CheckedElementCount(std::size_t rows, std::size_t columns);

void GenerateFp64Inputs(
    const MatrixShape &shape,
    std::uint64_t seed,
    std::vector<double> &aFp64,
    std::vector<double> &bFp64);

void ConvertFp64InputsToFp32(
    const std::vector<double> &aFp64,
    const std::vector<double> &bFp64,
    std::vector<float> &aFp32,
    std::vector<float> &bFp32);

std::uint16_t FloatToFp16Bits(float value);

float Fp16BitsToFloat(std::uint16_t value);

void ConvertFp64InputsToFp16(
    const std::vector<double> &aFp64,
    const std::vector<double> &bFp64,
    std::vector<std::uint16_t> &aFp16,
    std::vector<std::uint16_t> &bFp16);

void ConvertFp16ToFp32(
    const std::vector<std::uint16_t> &input,
    std::vector<float> &output);

ErrorMetrics ComputeErrorMetrics(
    const std::vector<double> &reference,
    const std::vector<float> &actual,
    double relativeDenominatorFloor = 1.0e-12);


std::string HumanReadableBytes(std::size_t bytes);

}  // namespace lowp

#endif  // LOW_PRECISION_GEMM_COMMON_H


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/include/gemm_fp64_cpu.h
#ifndef LOW_PRECISION_GEMM_FP64_CPU_H
#define LOW_PRECISION_GEMM_FP64_CPU_H

#include <cstddef>

#include "gemm_common.h"

namespace lowp {

struct CpuGemmConfig {
    std::size_t blockM = 32;
    std::size_t blockN = 64;
    std::size_t blockK = 64;
};

void CpuGemmFp64Serial(
    const double *a,
    const double *b,
    double *c,
    const MatrixShape &shape,
    const CpuGemmConfig &config = {});

}  // namespace lowp

#endif  // LOW_PRECISION_GEMM_FP64_CPU_H


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/include/vector_gemm_config.h
#ifndef LOW_PRECISION_GEMM_VECTOR_GEMM_CONFIG_H
#define LOW_PRECISION_GEMM_VECTOR_GEMM_CONFIG_H

#include <cstddef>
#include <cstdint>

#include "gemm_common.h"

namespace lowp {

enum class VectorPrecision {
    Fp32,
    Fp16
};

struct VectorGemmConfig {
    std::uint32_t blockDim = 0;
    std::uint32_t availableAicCores = 0;
    std::uint32_t availableAivCores = 0;
    std::uint32_t usedCoreCount = 0;
    std::uint32_t rowTile = 0;
    std::uint32_t columnTile = 0;
    std::uint32_t maximumReduction = 0;
    std::uint32_t alignmentElements = 0;
    std::size_t availableUbBytes = 0;
    std::size_t staticUbBytes = 0;
};

VectorGemmConfig GenerateVectorGemmConfig(
    const char *socVersion,
    const MatrixShape &shape,
    VectorPrecision precision);

}  // namespace lowp

#endif  // LOW_PRECISION_GEMM_VECTOR_GEMM_CONFIG_H


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/src/gemm_common.cpp
#include "gemm_common.h"

#include <algorithm>
#include <cmath>
#include <cstring>
#include <iomanip>
#include <iterator>
#include <limits>
#include <random>
#include <sstream>
#include <stdexcept>

namespace lowp {

namespace {

std::size_t CheckedMultiply(std::size_t lhs, std::size_t rhs)
{
    if (lhs != 0 && rhs > std::numeric_limits<std::size_t>::max() / lhs) {
        throw std::overflow_error("matrix element count exceeds size_t");
    }
    return lhs * rhs;
}

}  // namespace

std::size_t CheckedElementCount(std::size_t rows, std::size_t columns)
{
    return CheckedMultiply(rows, columns);
}

void GenerateFp64Inputs(
    const MatrixShape &shape,
    std::uint64_t seed,
    std::vector<double> &aFp64,
    std::vector<double> &bFp64)
{
    const std::size_t aElements = CheckedElementCount(shape.m, shape.k);
    const std::size_t bElements = CheckedElementCount(shape.k, shape.n);

    aFp64.resize(aElements);
    bFp64.resize(bElements);

    std::mt19937_64 generator(seed);
    std::uniform_real_distribution<double> distribution(-1.0, 1.0);

    for (std::size_t index = 0; index < aElements; ++index) {
        aFp64[index] = distribution(generator);
    }
    for (std::size_t index = 0; index < bElements; ++index) {
        bFp64[index] = distribution(generator);
    }
}

void ConvertFp64InputsToFp32(
    const std::vector<double> &aFp64,
    const std::vector<double> &bFp64,
    std::vector<float> &aFp32,
    std::vector<float> &bFp32)
{
    aFp32.assign(aFp64.begin(), aFp64.end());
    bFp32.assign(bFp64.begin(), bFp64.end());
}

std::uint16_t FloatToFp16Bits(float value)
{
    std::uint32_t bits = 0;
    static_assert(sizeof(bits) == sizeof(value));
    std::memcpy(&bits, &value, sizeof(bits));

    const std::uint16_t sign = static_cast<std::uint16_t>((bits >> 16U) & 0x8000U);
    const std::uint32_t exponent = (bits >> 23U) & 0xffU;
    const std::uint32_t mantissa = bits & 0x7fffffU;

    if (exponent == 0xffU) {
        if (mantissa == 0) {
            return static_cast<std::uint16_t>(sign | 0x7c00U);
        }
        // Preserve a payload and force a quiet half-precision NaN.
        std::uint16_t payload = static_cast<std::uint16_t>(mantissa >> 13U);
        payload = static_cast<std::uint16_t>(payload | 0x0200U);
        return static_cast<std::uint16_t>(sign | 0x7c00U | payload);
    }

    const std::int32_t halfExponent =
        static_cast<std::int32_t>(exponent) - 127 + 15;
    if (halfExponent >= 31) {
        return static_cast<std::uint16_t>(sign | 0x7c00U);
    }

    if (halfExponent <= 0) {
        if (halfExponent < -10) {
            return sign;
        }
        const std::uint32_t normalizedMantissa = mantissa | 0x800000U;
        const std::uint32_t shift =
            static_cast<std::uint32_t>(14 - halfExponent);
        std::uint32_t halfMantissa = normalizedMantissa >> shift;
        const std::uint32_t remainderMask = (1U << shift) - 1U;
        const std::uint32_t remainder = normalizedMantissa & remainderMask;
        const std::uint32_t halfway = 1U << (shift - 1U);
        if (remainder > halfway ||
            (remainder == halfway && (halfMantissa & 1U) != 0)) {
            ++halfMantissa;
        }
        return static_cast<std::uint16_t>(sign | halfMantissa);
    }

    std::uint32_t roundedExponent = static_cast<std::uint32_t>(halfExponent);
    std::uint32_t halfMantissa = mantissa >> 13U;
    const std::uint32_t remainder = mantissa & 0x1fffU;
    if (remainder > 0x1000U ||
        (remainder == 0x1000U && (halfMantissa & 1U) != 0)) {
        ++halfMantissa;
        if (halfMantissa == 0x400U) {
            halfMantissa = 0;
            ++roundedExponent;
            if (roundedExponent >= 31U) {
                return static_cast<std::uint16_t>(sign | 0x7c00U);
            }
        }
    }

    return static_cast<std::uint16_t>(
        sign | (roundedExponent << 10U) | halfMantissa);
}

float Fp16BitsToFloat(std::uint16_t value)
{
    const std::uint32_t sign =
        static_cast<std::uint32_t>(value & 0x8000U) << 16U;
    std::uint32_t exponent = (value >> 10U) & 0x1fU;
    std::uint32_t mantissa = value & 0x03ffU;
    std::uint32_t bits = 0;

    if (exponent == 0) {
        if (mantissa == 0) {
            bits = sign;
        } else {
            std::int32_t normalizedExponent = -14;
            while ((mantissa & 0x0400U) == 0) {
                mantissa <<= 1U;
                --normalizedExponent;
            }
            mantissa &= 0x03ffU;
            const std::uint32_t fp32Exponent =
                static_cast<std::uint32_t>(normalizedExponent + 127);
            bits = sign | (fp32Exponent << 23U) | (mantissa << 13U);
        }
    } else if (exponent == 0x1fU) {
        bits = sign | 0x7f800000U | (mantissa << 13U);
    } else {
        exponent = exponent - 15U + 127U;
        bits = sign | (exponent << 23U) | (mantissa << 13U);
    }

    float result = 0.0F;
    std::memcpy(&result, &bits, sizeof(result));
    return result;
}

void ConvertFp64InputsToFp16(
    const std::vector<double> &aFp64,
    const std::vector<double> &bFp64,
    std::vector<std::uint16_t> &aFp16,
    std::vector<std::uint16_t> &bFp16)
{
    aFp16.resize(aFp64.size());
    bFp16.resize(bFp64.size());
    std::transform(
        aFp64.begin(),
        aFp64.end(),
        aFp16.begin(),
        [](double value) { return FloatToFp16Bits(static_cast<float>(value)); });
    std::transform(
        bFp64.begin(),
        bFp64.end(),
        bFp16.begin(),
        [](double value) { return FloatToFp16Bits(static_cast<float>(value)); });
}

void ConvertFp16ToFp32(
    const std::vector<std::uint16_t> &input,
    std::vector<float> &output)
{
    output.resize(input.size());
    std::transform(
        input.begin(),
        input.end(),
        output.begin(),
        [](std::uint16_t value) { return Fp16BitsToFloat(value); });
}

ErrorMetrics ComputeErrorMetrics(
    const std::vector<double> &reference,
    const std::vector<float> &actual,
    double relativeDenominatorFloor)
{
    if (reference.size() != actual.size() || reference.empty()) {
        throw std::invalid_argument(
            "reference and actual results must have the same non-zero size");
    }
    if (!(relativeDenominatorFloor > 0.0) ||
        !std::isfinite(relativeDenominatorFloor)) {
        throw std::invalid_argument(
            "relative denominator floor must be finite and greater than zero");
    }

    ErrorMetrics metrics;
    metrics.relativeDenominatorFloor = relativeDenominatorFloor;
    for (std::size_t index = 0; index < reference.size(); ++index) {
        const double expected = reference[index];
        const double observed = static_cast<double>(actual[index]);
        if (!std::isfinite(expected) || !std::isfinite(observed)) {
            metrics.allFinite = false;
            metrics.maxAbsoluteError = std::numeric_limits<double>::infinity();
            metrics.maxRelativeError = std::numeric_limits<double>::infinity();
            continue;
        }

        const double absoluteError = std::abs(observed - expected);
        const double relativeError =
            absoluteError /
            std::max(std::abs(expected), relativeDenominatorFloor);
        metrics.maxAbsoluteError =
            std::max(metrics.maxAbsoluteError, absoluteError);
        metrics.maxRelativeError =
            std::max(metrics.maxRelativeError, relativeError);
    }
    return metrics;
}

std::string HumanReadableBytes(std::size_t bytes)
{
    static constexpr const char *units[] = {"B", "KiB", "MiB", "GiB", "TiB"};
    double value = static_cast<double>(bytes);
    std::size_t unitIndex = 0;
    while (value >= 1024.0 && unitIndex + 1 < std::size(units)) {
        value /= 1024.0;
        ++unitIndex;
    }

    std::ostringstream stream;
    stream << std::fixed << std::setprecision(unitIndex == 0 ? 0 : 2) << value << ' '
           << units[unitIndex];
    return stream.str();
}

}  // namespace lowp


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/src/gemm_fp64_cpu.cpp
#include "gemm_fp64_cpu.h"

#include <algorithm>
#include <stdexcept>

namespace lowp {

void CpuGemmFp64Serial(
    const double *a,
    const double *b,
    double *c,
    const MatrixShape &shape,
    const CpuGemmConfig &config)
{
    if (a == nullptr || b == nullptr || c == nullptr) {
        throw std::invalid_argument("CpuGemmFp64Serial received a null matrix pointer");
    }
    if (shape.m == 0 || shape.n == 0 || shape.k == 0) {
        throw std::invalid_argument("matrix dimensions must be greater than zero");
    }
    if (config.blockM == 0 || config.blockN == 0 || config.blockK == 0) {
        throw std::invalid_argument("CPU block dimensions must be greater than zero");
    }

    const std::size_t cElements = CheckedElementCount(shape.m, shape.n);
    std::fill(c, c + cElements, 0.0);

    // Cache-blocked but strictly single-threaded FP64 GEMM. Neither OpenMP nor
    // a threaded BLAS is used, so this remains the experiment's serial baseline.
    for (std::size_t rowBlock = 0; rowBlock < shape.m; rowBlock += config.blockM) {
        const std::size_t rowEnd = std::min(rowBlock + config.blockM, shape.m);
        for (std::size_t innerBlock = 0; innerBlock < shape.k; innerBlock += config.blockK) {
            const std::size_t innerEnd = std::min(innerBlock + config.blockK, shape.k);
            for (std::size_t columnBlock = 0; columnBlock < shape.n; columnBlock += config.blockN) {
                const std::size_t columnEnd = std::min(columnBlock + config.blockN, shape.n);
                for (std::size_t row = rowBlock; row < rowEnd; ++row) {
                    double *cRow = c + row * shape.n;
                    const double *aRow = a + row * shape.k;
                    for (std::size_t inner = innerBlock; inner < innerEnd; ++inner) {
                        const double aValue = aRow[inner];
                        const double *bRow = b + inner * shape.n;
                        for (std::size_t column = columnBlock; column < columnEnd; ++column) {
                            cRow[column] += aValue * bRow[column];
                        }
                    }
                }
            }
        }
    }
}

}  // namespace lowp


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/src/main_fp64_cpu.cpp
#include "gemm_common.h"
#include "gemm_fp64_cpu.h"

#include <algorithm>
#include <chrono>
#include <cstddef>
#include <cstdint>
#include <cstdlib>
#include <exception>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

namespace {

using Clock = std::chrono::steady_clock;

struct Options {
    lowp::MatrixShape shape{4096, 4096, 4096};
    lowp::CpuGemmConfig blocking;
    std::uint64_t seed = 20260717;
    std::size_t warmupRuns = 0;
    std::size_t measuredRuns = 1;
    bool printOutput = false;
};

std::size_t ParsePositiveSize(const std::string &text, const char *name)
{
    std::size_t consumed = 0;
    unsigned long long value = 0;
    try {
        value = std::stoull(text, &consumed, 10);
    } catch (const std::exception &) {
        throw std::invalid_argument(std::string(name) + " requires a positive integer");
    }
    if (consumed != text.size() || value == 0 ||
        value > static_cast<unsigned long long>(std::numeric_limits<std::size_t>::max())) {
        throw std::invalid_argument(std::string(name) + " requires a positive integer");
    }
    return static_cast<std::size_t>(value);
}

std::uint64_t ParseUnsigned64(const std::string &text, const char *name)
{
    if (text.empty() || text.front() == '-') {
        throw std::invalid_argument(std::string(name) + " requires an unsigned integer");
    }
    std::size_t consumed = 0;
    unsigned long long value = 0;
    try {
        value = std::stoull(text, &consumed, 10);
    } catch (const std::exception &) {
        throw std::invalid_argument(std::string(name) + " requires an unsigned integer");
    }
    if (consumed != text.size()) {
        throw std::invalid_argument(std::string(name) + " requires an unsigned integer");
    }
    return static_cast<std::uint64_t>(value);
}

void PrintHelp(const char *program)
{
    std::cout
        << "Usage: " << program << " [options]\n"
        << "  --m VALUE          rows of A and C (default 4096)\n"
        << "  --n VALUE          columns of B and C (default 4096)\n"
        << "  --k VALUE          columns of A / rows of B (default 4096)\n"
        << "  --seed VALUE       deterministic input seed (default 20260717)\n"
        << "  --warmup VALUE     untimed CPU runs (default 0)\n"
        << "  --repeat VALUE     measured CPU runs (default 1)\n"
        << "  --block-m VALUE    serial cache tile in M (default 32)\n"
        << "  --block-n VALUE    serial cache tile in N (default 64)\n"
        << "  --block-k VALUE    serial cache tile in K (default 64)\n"
        << "  --print-output     print the first eight output elements\n"
        << "  -h, --help         show this message\n";
}

Options ParseOptions(int argc, char **argv)
{
    Options options;
    for (int index = 1; index < argc; ++index) {
        const std::string option = argv[index];
        if (option == "-h" || option == "--help") {
            PrintHelp(argv[0]);
            std::exit(EXIT_SUCCESS);
        }
        if (option == "--print-output") {
            options.printOutput = true;
            continue;
        }
        if (index + 1 >= argc) {
            throw std::invalid_argument("missing value after " + option);
        }
        const std::string value = argv[++index];
        if (option == "--m") {
            options.shape.m = ParsePositiveSize(value, "--m");
        } else if (option == "--n") {
            options.shape.n = ParsePositiveSize(value, "--n");
        } else if (option == "--k") {
            options.shape.k = ParsePositiveSize(value, "--k");
        } else if (option == "--seed") {
            options.seed = ParseUnsigned64(value, "--seed");
        } else if (option == "--warmup") {
            options.warmupRuns = value == "0" ? 0 : ParsePositiveSize(value, "--warmup");
        } else if (option == "--repeat") {
            options.measuredRuns = ParsePositiveSize(value, "--repeat");
        } else if (option == "--block-m") {
            options.blocking.blockM = ParsePositiveSize(value, "--block-m");
        } else if (option == "--block-n") {
            options.blocking.blockN = ParsePositiveSize(value, "--block-n");
        } else if (option == "--block-k") {
            options.blocking.blockK = ParsePositiveSize(value, "--block-k");
        } else {
            throw std::invalid_argument("unknown option: " + option);
        }
    }
    return options;
}

double Milliseconds(Clock::duration duration)
{
    return std::chrono::duration<double, std::milli>(duration).count();
}

}  // namespace

int main(int argc, char **argv)
{
    try {
        const Options options = ParseOptions(argc, argv);
        const std::size_t cElements =
            lowp::CheckedElementCount(options.shape.m, options.shape.n);

        std::vector<double> a;
        std::vector<double> b;
        std::vector<double> c(cElements);
        lowp::GenerateFp64Inputs(options.shape, options.seed, a, b);

        for (std::size_t run = 0; run < options.warmupRuns; ++run) {
            lowp::CpuGemmFp64Serial(
                a.data(), b.data(), c.data(), options.shape, options.blocking);
        }

        std::vector<double> runMilliseconds;
        runMilliseconds.reserve(options.measuredRuns);
        for (std::size_t run = 0; run < options.measuredRuns; ++run) {
            const auto start = Clock::now();
            lowp::CpuGemmFp64Serial(
                a.data(), b.data(), c.data(), options.shape, options.blocking);
            const auto end = Clock::now();
            runMilliseconds.push_back(Milliseconds(end - start));
        }

        const double averageMs =
            std::accumulate(runMilliseconds.begin(), runMilliseconds.end(), 0.0) /
            static_cast<double>(runMilliseconds.size());

        std::cout << "[CPU FP64 serial baseline]\n"
                  << "  shape M/N/K      : " << options.shape.m << '/' << options.shape.n
                  << '/' << options.shape.k << '\n'
                  << "  cache tile M/N/K : " << options.blocking.blockM << '/'
                  << options.blocking.blockN << '/' << options.blocking.blockK << '\n'
                  << "  warmup/repeat    : " << options.warmupRuns << '/'
                  << options.measuredRuns << '\n'
                  << std::fixed << std::setprecision(3)
                  << "  total time       : " << averageMs << " ms\n";

        if (options.printOutput) {
            const std::size_t count = std::min<std::size_t>(8, c.size());
            std::cout << "  output sample    :";
            for (std::size_t index = 0; index < count; ++index) {
                std::cout << ' ' << c[index];
            }
            std::cout << '\n';
        }
        return EXIT_SUCCESS;
    } catch (const std::exception &error) {
        std::cerr << "[fatal] " << error.what() << '\n';
        return EXIT_FAILURE;
    }
}


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/host_launch/acl_runtime.h
#ifndef LOW_PRECISION_GEMM_ACL_RUNTIME_H
#define LOW_PRECISION_GEMM_ACL_RUNTIME_H

#include <cstddef>
#include <cstdint>
#include <sstream>
#include <stdexcept>

#include "acl/acl.h"

namespace lowp {

inline void CheckAcl(aclError code, const char *expression, const char *file, int line)
{
    if (code != ACL_ERROR_NONE) {
        std::ostringstream message;
        message << file << ':' << line << ": " << expression << " failed with aclError=" << code;
        throw std::runtime_error(message.str());
    }
}

class AclSession {
public:
    explicit AclSession(std::uint32_t deviceId) : deviceId_(deviceId)
    {
        try {
            CheckAcl(aclInit(nullptr), "aclInit(nullptr)", __FILE__, __LINE__);
            initialized_ = true;
            CheckAcl(
                aclrtSetDevice(static_cast<int32_t>(deviceId_)),
                "aclrtSetDevice(deviceId)",
                __FILE__,
                __LINE__);
            deviceSet_ = true;
            CheckAcl(aclrtCreateStream(&stream_), "aclrtCreateStream(&stream)", __FILE__, __LINE__);
        } catch (...) {
            Cleanup();
            throw;
        }
    }

    AclSession(const AclSession &) = delete;
    AclSession &operator=(const AclSession &) = delete;

    ~AclSession()
    {
        Cleanup();
    }

    aclrtStream Stream() const
    {
        return stream_;
    }

private:
    void Cleanup() noexcept
    {
        if (stream_ != nullptr) {
            (void)aclrtDestroyStream(stream_);
            stream_ = nullptr;
        }
        if (deviceSet_) {
            (void)aclrtResetDevice(static_cast<int32_t>(deviceId_));
            deviceSet_ = false;
        }
        if (initialized_) {
            (void)aclFinalize();
            initialized_ = false;
        }
    }

    std::uint32_t deviceId_;
    bool initialized_ = false;
    bool deviceSet_ = false;
    aclrtStream stream_ = nullptr;
};

class DeviceBuffer {
public:
    explicit DeviceBuffer(std::size_t bytes) : bytes_(bytes)
    {
        if (bytes_ != 0) {
            CheckAcl(
                aclrtMalloc(
                    reinterpret_cast<void **>(&pointer_),
                    bytes_,
                    ACL_MEM_MALLOC_HUGE_FIRST),
                "aclrtMalloc",
                __FILE__,
                __LINE__);
        }
    }

    DeviceBuffer(const DeviceBuffer &) = delete;
    DeviceBuffer &operator=(const DeviceBuffer &) = delete;

    ~DeviceBuffer()
    {
        if (pointer_ != nullptr) {
            (void)aclrtFree(pointer_);
        }
    }

    std::uint8_t *Data() const
    {
        return pointer_;
    }

    std::size_t Size() const
    {
        return bytes_;
    }

private:
    std::uint8_t *pointer_ = nullptr;
    std::size_t bytes_ = 0;
};

class PinnedHostBuffer {
public:
    explicit PinnedHostBuffer(std::size_t bytes) : bytes_(bytes)
    {
        if (bytes_ != 0) {
            CheckAcl(
                aclrtMallocHost(reinterpret_cast<void **>(&pointer_), bytes_),
                "aclrtMallocHost",
                __FILE__,
                __LINE__);
        }
    }

    PinnedHostBuffer(const PinnedHostBuffer &) = delete;
    PinnedHostBuffer &operator=(const PinnedHostBuffer &) = delete;

    ~PinnedHostBuffer()
    {
        if (pointer_ != nullptr) {
            (void)aclrtFreeHost(pointer_);
        }
    }

    std::uint8_t *Data() const
    {
        return pointer_;
    }

    std::size_t Size() const
    {
        return bytes_;
    }

private:
    std::uint8_t *pointer_ = nullptr;
    std::size_t bytes_ = 0;
};

}  // namespace lowp

#define LOWP_ACL_CHECK(expression) \
    ::lowp::CheckAcl((expression), #expression, __FILE__, __LINE__)

#endif  // LOW_PRECISION_GEMM_ACL_RUNTIME_H


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/host_launch/vector_gemm_config.cpp
#include "vector_gemm_config.h"

#include <algorithm>
#include <limits>
#include <sstream>
#include <stdexcept>
#include <string>

#include "tiling/platform/platform_ascendc.h"

namespace lowp {

namespace {

constexpr std::uint32_t kMaximumReduction = 4096;
constexpr std::uint32_t kColumnTile = 1024;
constexpr std::uint32_t kFp32RowTile = 8;
constexpr std::uint32_t kFp16RowTile = 16;

std::uint32_t ToKernelDimension(std::size_t value, const char *name)
{
    if (value == 0 ||
        value > static_cast<std::size_t>(std::numeric_limits<std::uint32_t>::max())) {
        std::ostringstream message;
        message << name << " must be in [1, UINT32_MAX]";
        throw std::invalid_argument(message.str());
    }
    return static_cast<std::uint32_t>(value);
}

}  // namespace

VectorGemmConfig GenerateVectorGemmConfig(
    const char *socVersion,
    const MatrixShape &shape,
    VectorPrecision precision)
{
    if (socVersion == nullptr || socVersion[0] == '\0') {
        throw std::invalid_argument("SOC_VERSION must not be empty");
    }

    auto *platform = platform_ascendc::PlatformAscendCManager::GetInstance(socVersion);
    if (platform == nullptr) {
        throw std::runtime_error(
            std::string("failed to obtain Ascend platform information for ") + socVersion);
    }

    const std::uint32_t m = ToKernelDimension(shape.m, "M");
    const std::uint32_t n = ToKernelDimension(shape.n, "N");
    const std::uint32_t k = ToKernelDimension(shape.k, "K");

    VectorGemmConfig config;
    config.availableAicCores = platform->GetCoreNumAic();
    config.availableAivCores = platform->GetCoreNumAiv();
    if (config.availableAivCores == 0) {
        throw std::runtime_error("the selected SoC reports zero AI Vector cores");
    }

    const std::uint32_t elementBytes =
        precision == VectorPrecision::Fp32 ? sizeof(float) : sizeof(std::uint16_t);
    config.rowTile =
        precision == VectorPrecision::Fp32 ? kFp32RowTile : kFp16RowTile;
    config.columnTile = kColumnTile;
    config.maximumReduction = kMaximumReduction;
    config.alignmentElements = 32U / elementBytes;

    if (k > config.maximumReduction) {
        std::ostringstream message;
        message << "K=" << k << " exceeds the static Vector kernel capacity "
                << config.maximumReduction;
        throw std::invalid_argument(message.str());
    }
    if (n % config.alignmentElements != 0 ||
        k % config.alignmentElements != 0) {
        std::ostringstream message;
        message << "Vector DataCopy requires N and K to be multiples of "
                << config.alignmentElements << " elements for this "
                << (precision == VectorPrecision::Fp32 ? "FP32" : "FP16")
                << " kernel";
        throw std::invalid_argument(message.str());
    }

    const std::uint64_t rowTileCount =
        (static_cast<std::uint64_t>(m) + config.rowTile - 1U) / config.rowTile;
    config.usedCoreCount = static_cast<std::uint32_t>(
        std::min<std::uint64_t>(rowTileCount, config.availableAivCores));
    config.blockDim = config.usedCoreCount;

    // aLocal + bLocal + aBroadcastLocal + productLocal + cLocal
    const std::uint64_t localElements =
        static_cast<std::uint64_t>(config.rowTile) * config.maximumReduction +
        3ULL * config.columnTile +
        static_cast<std::uint64_t>(config.rowTile) * config.columnTile;
    config.staticUbBytes = static_cast<std::size_t>(localElements * elementBytes);

    std::uint64_t availableUbBytes = 0;
    platform->GetCoreMemSize(
        platform_ascendc::CoreMemType::UB,
        availableUbBytes);
    config.availableUbBytes = static_cast<std::size_t>(availableUbBytes);
    if (config.staticUbBytes > config.availableUbBytes) {
        std::ostringstream message;
        message << "static Vector kernel needs " << config.staticUbBytes
                << " UB bytes per core, but " << socVersion << " reports "
                << config.availableUbBytes;
        throw std::runtime_error(message.str());
    }

    return config;
}

}  // namespace lowp


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/host_launch/vector_gemm_main_impl.h
#ifndef LOW_PRECISION_GEMM_VECTOR_GEMM_MAIN_IMPL_H
#define LOW_PRECISION_GEMM_VECTOR_GEMM_MAIN_IMPL_H

#include "acl_runtime.h"
#include "gemm_common.h"
#include "gemm_fp64_cpu.h"
#include "vector_gemm_config.h"

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdlib>
#include <cstring>
#include <exception>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <sstream>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(LOWP_VECTOR_GEMM_FP32) == defined(LOWP_VECTOR_GEMM_FP16)
#error "Define exactly one of LOWP_VECTOR_GEMM_FP32 or LOWP_VECTOR_GEMM_FP16"
#endif

#if defined(LOWP_VECTOR_GEMM_FP32)
#include "aclrtlaunch_fp32_vector_gemm.h"
#else
#include "aclrtlaunch_fp16_vector_gemm.h"
#endif

namespace lowp_host {

namespace {

using Clock = std::chrono::steady_clock;

#if defined(LOWP_VECTOR_GEMM_FP32)
using NpuStorage = float;
constexpr const char *kPrecisionUpper = "FP32";
constexpr const char *kPrecisionLower = "fp32";
constexpr const char *kDefaultJsonPath = "results/fp32_result.json";
constexpr lowp::VectorPrecision kVectorPrecision = lowp::VectorPrecision::Fp32;
#else
using NpuStorage = std::uint16_t;
constexpr const char *kPrecisionUpper = "FP16";
constexpr const char *kPrecisionLower = "fp16";
constexpr const char *kDefaultJsonPath = "results/fp16_result.json";
constexpr lowp::VectorPrecision kVectorPrecision = lowp::VectorPrecision::Fp16;
#endif

struct Options {
    lowp::MatrixShape shape{4096, 4096, 4096};
    std::uint64_t seed = 20260717;
    std::uint32_t deviceId = 0;
    std::size_t warmupRuns = 5;
    std::size_t measuredRuns = 30;
    double relativeDenominatorFloor = 1.0e-12;
    std::string jsonPath = kDefaultJsonPath;
};

struct NpuTiming {
    double hostToDeviceMs = 0.0;
    double kernelAverageMs = 0.0;
    double deviceToHostMs = 0.0;
    double endToEndOneRunMs = 0.0;
};

struct ExperimentResult {
    double cpuMilliseconds = 0.0;
    NpuTiming npu;
    double endToEndSpeedup = 0.0;
    lowp::ErrorMetrics errors;
};

double Milliseconds(Clock::duration duration)
{
    return std::chrono::duration<double, std::milli>(duration).count();
}

std::size_t ParsePositiveSize(const std::string &text, const char *optionName)
{
    std::size_t consumed = 0;
    unsigned long long value = 0;
    try {
        value = std::stoull(text, &consumed, 10);
    } catch (const std::exception &) {
        throw std::invalid_argument(std::string(optionName) + " requires a positive integer");
    }
    if (consumed != text.size() || value == 0 ||
        value > static_cast<unsigned long long>(std::numeric_limits<std::size_t>::max())) {
        throw std::invalid_argument(std::string(optionName) + " requires a positive integer");
    }
    return static_cast<std::size_t>(value);
}

std::uint64_t ParseUnsigned64(const std::string &text, const char *optionName)
{
    if (text.empty() || text.front() == '-') {
        throw std::invalid_argument(std::string(optionName) + " requires an unsigned integer");
    }
    std::size_t consumed = 0;
    unsigned long long value = 0;
    try {
        value = std::stoull(text, &consumed, 10);
    } catch (const std::exception &) {
        throw std::invalid_argument(std::string(optionName) + " requires an unsigned integer");
    }
    if (consumed != text.size()) {
        throw std::invalid_argument(std::string(optionName) + " requires an unsigned integer");
    }
    return static_cast<std::uint64_t>(value);
}

double ParsePositiveDouble(const std::string &text, const char *optionName)
{
    std::size_t consumed = 0;
    double value = 0.0;
    try {
        value = std::stod(text, &consumed);
    } catch (const std::exception &) {
        throw std::invalid_argument(std::string(optionName) + " requires a positive number");
    }
    if (consumed != text.size() || !(value > 0.0) || !std::isfinite(value)) {
        throw std::invalid_argument(std::string(optionName) + " requires a positive number");
    }
    return value;
}

void PrintHelp(const char *program)
{
    std::cout
        << "Usage: " << program << " [options]\n"
        << "  --m VALUE             rows of A and C (default 4096)\n"
        << "  --n VALUE             columns of B and C (default 4096)\n"
        << "  --k VALUE             columns of A / rows of B (default 4096)\n"
        << "  --seed VALUE          deterministic input seed (default 20260717)\n"
        << "  --device VALUE        Ascend logical device id (default 0)\n"
        << "  --warmup VALUE        NPU warmup launches (default 5)\n"
        << "  --repeat VALUE        measured NPU launches (default 30)\n"
        << "  --relative-floor X    max-relative-error denominator floor (default 1e-12)\n"
        << "  --json PATH           JSON result path (default " << kDefaultJsonPath << ")\n"
        << "  --help                show this message\n";
}

Options ParseOptions(int argc, char **argv)
{
    Options options;
    for (int index = 1; index < argc; ++index) {
        const std::string option = argv[index];
        if (option == "--help") {
            PrintHelp(argv[0]);
            std::exit(EXIT_SUCCESS);
        }
        if (index + 1 >= argc) {
            throw std::invalid_argument("missing value after " + option);
        }
        const std::string value = argv[++index];
        if (option == "--m") {
            options.shape.m = ParsePositiveSize(value, "--m");
        } else if (option == "--n") {
            options.shape.n = ParsePositiveSize(value, "--n");
        } else if (option == "--k") {
            options.shape.k = ParsePositiveSize(value, "--k");
        } else if (option == "--seed") {
            options.seed = ParseUnsigned64(value, "--seed");
        } else if (option == "--device") {
            const std::uint64_t parsed = ParseUnsigned64(value, "--device");
            if (parsed > std::numeric_limits<std::uint32_t>::max()) {
                throw std::invalid_argument("--device is out of range");
            }
            options.deviceId = static_cast<std::uint32_t>(parsed);
        } else if (option == "--warmup") {
            options.warmupRuns =
                value == "0" ? 0 : ParsePositiveSize(value, "--warmup");
        } else if (option == "--repeat") {
            options.measuredRuns = ParsePositiveSize(value, "--repeat");
        } else if (option == "--relative-floor") {
            options.relativeDenominatorFloor =
                ParsePositiveDouble(value, "--relative-floor");
        } else if (option == "--json") {
            if (value.empty()) {
                throw std::invalid_argument("--json path must not be empty");
            }
            options.jsonPath = value;
        } else {
            throw std::invalid_argument("unknown option: " + option);
        }
    }
    return options;
}

std::size_t CheckedByteCount(std::size_t elements, std::size_t elementBytes)
{
    if (elements > std::numeric_limits<std::size_t>::max() / elementBytes) {
        throw std::overflow_error("matrix byte count exceeds size_t");
    }
    return elements * elementBytes;
}

void ConvertInputs(
    const std::vector<double> &aFp64,
    const std::vector<double> &bFp64,
    std::vector<NpuStorage> &aNpu,
    std::vector<NpuStorage> &bNpu)
{
#if defined(LOWP_VECTOR_GEMM_FP32)
    lowp::ConvertFp64InputsToFp32(aFp64, bFp64, aNpu, bNpu);
#else
    lowp::ConvertFp64InputsToFp16(aFp64, bFp64, aNpu, bNpu);
#endif
}

void ConvertOutputForComparison(
    const std::vector<NpuStorage> &npuOutput,
    std::vector<float> &comparisonOutput)
{
#if defined(LOWP_VECTOR_GEMM_FP32)
    comparisonOutput.assign(npuOutput.begin(), npuOutput.end());
#else
    lowp::ConvertFp16ToFp32(npuOutput, comparisonOutput);
#endif
}

void LaunchVectorGemm(
    const lowp::VectorGemmConfig &config,
    const lowp::MatrixShape &shape,
    aclrtStream stream,
    const lowp::DeviceBuffer &a,
    const lowp::DeviceBuffer &b,
    const lowp::DeviceBuffer &c)
{
    const auto m = static_cast<std::uint32_t>(shape.m);
    const auto n = static_cast<std::uint32_t>(shape.n);
    const auto k = static_cast<std::uint32_t>(shape.k);
#if defined(LOWP_VECTOR_GEMM_FP32)
    ACLRT_LAUNCH_KERNEL(fp32_vector_gemm)(
        config.blockDim, stream, a.Data(), b.Data(), c.Data(), m, n, k);
#else
    ACLRT_LAUNCH_KERNEL(fp16_vector_gemm)(
        config.blockDim, stream, a.Data(), b.Data(), c.Data(), m, n, k);
#endif
}

NpuTiming RunNpu(
    const std::vector<NpuStorage> &a,
    const std::vector<NpuStorage> &b,
    std::vector<NpuStorage> &c,
    const lowp::VectorGemmConfig &config,
    const Options &options)
{
    const std::size_t aBytes = CheckedByteCount(a.size(), sizeof(NpuStorage));
    const std::size_t bBytes = CheckedByteCount(b.size(), sizeof(NpuStorage));
    const std::size_t cBytes = CheckedByteCount(c.size(), sizeof(NpuStorage));

    lowp::AclSession session(options.deviceId);
    lowp::PinnedHostBuffer aHost(aBytes);
    lowp::PinnedHostBuffer bHost(bBytes);
    lowp::PinnedHostBuffer cHost(cBytes);
    std::memcpy(aHost.Data(), a.data(), aBytes);
    std::memcpy(bHost.Data(), b.data(), bBytes);

    lowp::DeviceBuffer aDevice(aBytes);
    lowp::DeviceBuffer bDevice(bBytes);
    lowp::DeviceBuffer cDevice(cBytes);

    const Clock::time_point h2dStart = Clock::now();
    LOWP_ACL_CHECK(aclrtMemcpy(
        aDevice.Data(), aDevice.Size(), aHost.Data(), aHost.Size(),
        ACL_MEMCPY_HOST_TO_DEVICE));
    LOWP_ACL_CHECK(aclrtMemcpy(
        bDevice.Data(), bDevice.Size(), bHost.Data(), bHost.Size(),
        ACL_MEMCPY_HOST_TO_DEVICE));
    const Clock::time_point h2dEnd = Clock::now();

    for (std::size_t run = 0; run < options.warmupRuns; ++run) {
        LaunchVectorGemm(
            config, options.shape, session.Stream(), aDevice, bDevice, cDevice);
        LOWP_ACL_CHECK(aclrtSynchronizeStream(session.Stream()));
    }

    std::vector<double> kernelMilliseconds;
    kernelMilliseconds.reserve(options.measuredRuns);
    for (std::size_t run = 0; run < options.measuredRuns; ++run) {
        const Clock::time_point kernelStart = Clock::now();
        LaunchVectorGemm(
            config, options.shape, session.Stream(), aDevice, bDevice, cDevice);
        LOWP_ACL_CHECK(aclrtSynchronizeStream(session.Stream()));
        const Clock::time_point kernelEnd = Clock::now();
        kernelMilliseconds.push_back(Milliseconds(kernelEnd - kernelStart));
    }

    const Clock::time_point d2hStart = Clock::now();
    LOWP_ACL_CHECK(aclrtMemcpy(
        cHost.Data(), cHost.Size(), cDevice.Data(), cDevice.Size(),
        ACL_MEMCPY_DEVICE_TO_HOST));
    const Clock::time_point d2hEnd = Clock::now();
    std::memcpy(c.data(), cHost.Data(), cBytes);

    NpuTiming timing;
    timing.hostToDeviceMs = Milliseconds(h2dEnd - h2dStart);
    timing.deviceToHostMs = Milliseconds(d2hEnd - d2hStart);
    timing.kernelAverageMs =
        std::accumulate(kernelMilliseconds.begin(), kernelMilliseconds.end(), 0.0) /
        static_cast<double>(kernelMilliseconds.size());
    timing.endToEndOneRunMs =
        timing.hostToDeviceMs + timing.kernelAverageMs + timing.deviceToHostMs;
    return timing;
}

std::string JsonNumber(double value)
{
    if (!std::isfinite(value)) {
        return "null";
    }
    std::ostringstream stream;
    stream << std::setprecision(17) << value;
    return stream.str();
}

void WriteJson(
    const std::string &path,
    const Options &options,
    const ExperimentResult &result)
{
    std::ofstream output(path);
    if (!output.is_open()) {
        throw std::runtime_error("failed to open JSON result path: " + path);
    }

    output
        << "{\n"
        << "  \"experiment\": \"CPU FP64 serial vs NPU " << kPrecisionUpper
        << " GEMM\",\n"
        << "  \"precision\": \"" << kPrecisionUpper << "\",\n"
        << "  \"shape\": {\"m\": " << options.shape.m
        << ", \"n\": " << options.shape.n << ", \"k\": " << options.shape.k << "},\n"
        << "  \"metrics\": {\n"
        << "    \"end_to_end_total_ms\": "
        << JsonNumber(result.npu.endToEndOneRunMs) << ",\n"
        << "    \"speedup\": " << JsonNumber(result.endToEndSpeedup) << ",\n"
        << "    \"max_absolute_error\": "
        << JsonNumber(result.errors.maxAbsoluteError) << ",\n"
        << "    \"max_relative_error\": "
        << JsonNumber(result.errors.maxRelativeError) << "\n"
        << "  }\n"
        << "}\n";
}

void PrintConfiguration(
    const Options &options,
    const lowp::VectorGemmConfig &config,
    std::size_t hostBytes)
{
    std::cout
        << "[configuration]\n"
        << "  SoC version              : " << SOC_VERSION << '\n'
        << "  matrix shape             : M=" << options.shape.m
        << ", N=" << options.shape.n << ", K=" << options.shape.k << '\n'
        << "  deterministic seed       : " << options.seed << '\n'
        << "  NPU precision            : " << kPrecisionUpper << '\n'
        << "  programming method      : static Tensor\n"
        << "  compute unit            : AI Vector Core\n"
        << "  arithmetic sequence      : Mul -> typed product Tensor -> Add\n"
        << "  estimated host matrices  : " << lowp::HumanReadableBytes(hostBytes) << '\n'
        << "  AIC/AIV cores reported   : " << config.availableAicCores
        << '/' << config.availableAivCores << '\n'
        << "  launch/used AIV cores    : " << config.blockDim
        << '/' << config.usedCoreCount << '\n'
        << "  row/column tile          : " << config.rowTile
        << " x " << config.columnTile << '\n'
        << "  static UB per core       : "
        << lowp::HumanReadableBytes(config.staticUbBytes) << " / "
        << lowp::HumanReadableBytes(config.availableUbBytes) << "\n\n";
}

void PrintResult(const ExperimentResult &result)
{
    std::cout << std::fixed << std::setprecision(3)
              << "[experiment metrics]\n"
              << "  end-to-end total time    : "
              << result.npu.endToEndOneRunMs << " ms\n"
              << "  speedup                  : "
              << result.endToEndSpeedup << "x\n"
              << std::scientific << std::setprecision(8)
              << "  max absolute error       : "
              << result.errors.maxAbsoluteError << '\n'
              << "  max relative error       : "
              << result.errors.maxRelativeError << '\n';
}

}  // namespace

int RunVectorGemmExperiment(int argc, char **argv)
{
    try {
        const Options options = ParseOptions(argc, argv);
        const std::size_t aElements =
            lowp::CheckedElementCount(options.shape.m, options.shape.k);
        const std::size_t bElements =
            lowp::CheckedElementCount(options.shape.k, options.shape.n);
        const std::size_t cElements =
            lowp::CheckedElementCount(options.shape.m, options.shape.n);

        const lowp::VectorGemmConfig config =
            lowp::GenerateVectorGemmConfig(
                SOC_VERSION, options.shape, kVectorPrecision);

        std::vector<double> aFp64;
        std::vector<double> bFp64;
        std::vector<double> cFp64(cElements);
        std::vector<NpuStorage> aNpu;
        std::vector<NpuStorage> bNpu;
        std::vector<NpuStorage> cNpu(cElements);
        std::vector<float> cForComparison;

        const std::size_t hostBytes =
            CheckedByteCount(aElements, sizeof(double)) +
            CheckedByteCount(bElements, sizeof(double)) +
            CheckedByteCount(cElements, sizeof(double)) +
            CheckedByteCount(aElements, sizeof(NpuStorage)) +
            CheckedByteCount(bElements, sizeof(NpuStorage)) +
            CheckedByteCount(cElements, sizeof(NpuStorage)) +
            CheckedByteCount(cElements, sizeof(float));
        PrintConfiguration(options, config, hostBytes);

        std::cout << "[stage] generating FP64 inputs and explicit "
                  << kPrecisionUpper << " conversions..." << std::endl;
        lowp::GenerateFp64Inputs(options.shape, options.seed, aFp64, bFp64);
        ConvertInputs(aFp64, bFp64, aNpu, bNpu);

        std::cout << "[stage] running single-threaded CPU FP64 baseline..."
                  << std::endl;
        const Clock::time_point cpuStart = Clock::now();
        lowp::CpuGemmFp64Serial(
            aFp64.data(), bFp64.data(), cFp64.data(), options.shape);
        const Clock::time_point cpuEnd = Clock::now();

        std::cout << "[stage] running NPU " << kPrecisionUpper
                  << " GEMM based on Vector basic APIs..." << std::endl;
        ExperimentResult result;
        result.cpuMilliseconds = Milliseconds(cpuEnd - cpuStart);
        result.npu = RunNpu(aNpu, bNpu, cNpu, config, options);
        ConvertOutputForComparison(cNpu, cForComparison);
        result.errors = lowp::ComputeErrorMetrics(
            cFp64, cForComparison, options.relativeDenominatorFloor);

        result.endToEndSpeedup =
            result.cpuMilliseconds / result.npu.endToEndOneRunMs;

        PrintResult(result);
        WriteJson(options.jsonPath, options, result);
        std::cout << "\n[result] JSON written to "
                  << options.jsonPath << std::endl;

        return result.errors.allFinite ? EXIT_SUCCESS : EXIT_FAILURE;
    } catch (const std::exception &error) {
        std::cerr << "[fatal] " << error.what() << std::endl;
        return EXIT_FAILURE;
    }
}

}  // namespace lowp_host

#endif  // LOW_PRECISION_GEMM_VECTOR_GEMM_MAIN_IMPL_H


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/host_launch/main_fp32_npu.cpp
#define LOWP_VECTOR_GEMM_FP32
#include "vector_gemm_main_impl.h"

int main(int argc, char **argv)
{
    return lowp_host::RunVectorGemmExperiment(argc, argv);
}


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/host_launch/main_fp16_npu.cpp
#define LOWP_VECTOR_GEMM_FP16
#include "vector_gemm_main_impl.h"

int main(int argc, char **argv)
{
    return lowp_host::RunVectorGemmExperiment(argc, argv);
}


### 2.3 工程公共文件检查

公共文件写入完成后，检查关键文件是否已经生成。检查结果均为`OK`时，说明Host侧工程骨架已经就绪。文件大小检查还可以识别空文件或写入过程被意外中断的情况。


In [ ]:
public_files = [
    WORK_DIR / "include" / "gemm_common.h",
    WORK_DIR / "include" / "gemm_fp64_cpu.h",
    WORK_DIR / "include" / "vector_gemm_config.h",
    WORK_DIR / "src" / "gemm_common.cpp",
    WORK_DIR / "src" / "gemm_fp64_cpu.cpp",
    WORK_DIR / "src" / "main_fp64_cpu.cpp",
    WORK_DIR / "ascend_ops" / "host_launch" / "acl_runtime.h",
    WORK_DIR / "ascend_ops" / "host_launch" / "vector_gemm_config.cpp",
    WORK_DIR / "ascend_ops" / "host_launch" / "vector_gemm_main_impl.h",
    WORK_DIR / "ascend_ops" / "host_launch" / "main_fp32_npu.cpp",
    WORK_DIR / "ascend_ops" / "host_launch" / "main_fp16_npu.cpp",
]

for path in public_files:
    state = "OK" if path.exists() and path.stat().st_size > 0 else "MISSING"
    size = path.stat().st_size if path.exists() else 0
    print(f"{state:7s} {size:7d} bytes  {path.relative_to(WORK_DIR)}")

assert all(path.exists() and path.stat().st_size > 0 for path in public_files)


---
## 3. 问题分析

本节依次分析GEMM的输入输出、数据类型与存储方式、矩阵分块与任务划分、Vector计算过程、Host侧与Device侧协同执行方式以及实验参数设置。后续核函数开发将依据本节确定的数据组织和计算流程展开。


### 3.1 输入输出与计算公式

本实验实现稠密矩阵乘法$C=A\times B$。输入矩阵$A$的形状为$M\times K$，输入矩阵$B$的形状为$K\times N$，输出矩阵$C$的形状为$M\times N$：

$$
A\in\mathbb{R}^{M\times K},\qquad
B\in\mathbb{R}^{K\times N},\qquad
C\in\mathbb{R}^{M\times N}
$$

输出矩阵第$i$行、第$j$列的元素为：

$$
C_{i,j}=\sum_{p=0}^{K-1}A_{i,p}\times B_{p,j}
$$

$K$为乘加计算的归约维度。按照一次乘法和一次加法分别计为一次浮点运算，GEMM的计算量近似为：

$$
2MNK
$$

三个矩阵均采用行主序连续存储。元素$A_{i,p}$、$B_{p,j}$和$C_{i,j}$在一维存储空间中的偏移量分别为`i * K + p`、`p * N + j`和`i * N + j`。CPU与NPU程序必须使用相同的矩阵形状、随机种子和存储顺序。


### 3.2 数据类型与数据布局

本实验使用FP64、FP32和FP16三种浮点数据类型。三种类型的存储开销和精度特征如下表所示。

| 数据类型 | 单元素存储量 | 指数位 | 显式尾数位 | 本实验中的用途 |
|---|---:|---:|---:|---|
| FP64 | 8字节 | 11 | 52 | CPU输入、计算和参考结果 |
| FP32 | 4字节 | 8 | 23 | NPU FP32输入、计算和输出 |
| FP16 | 2字节 | 5 | 10 | NPU FP16输入、计算和输出 |

Host侧首先使用固定随机种子生成FP64矩阵$A$和$B$，再由同一份FP64数据分别转换得到FP32和FP16输入。这样可以保证两种NPU实现具有一致的数据来源。FP16数据在Host侧以16位位模式保存，传入NPU程序后按`aclFloat16`解释。

设目标数据类型的单元素存储量为$s$字节，仅统计矩阵$A$、$B$和$C$时，所需存储空间为：

$$
S=s(MK+KN+MN)
$$

与FP64相比，FP32的矩阵存储量约为其二分之一，FP16约为其四分之一。降低数据精度可以减少存储空间和数据搬运量，但精度转换以及后续乘加计算均可能产生舍入误差。FP16的有效位数较少，随着归约维度$K$增大，累加误差通常更为明显。


In [ ]:
# 估算主实验规模的计算量与矩阵存储空间
M = N = K = 4096
bytes_per_type = {"FP64": 8, "FP32": 4, "FP16": 2}
matrix_elements = M * K + K * N + M * N
operations = 2 * M * N * K

print(f"矩阵规模：M={M}, N={N}, K={K}")
print(f"浮点运算次数：{operations:,}（约{operations / 1e9:.3f}十亿次）")
for name, item_bytes in bytes_per_type.items():
    total = matrix_elements * item_bytes
    print(f"{name}矩阵A、B、C的存储空间：{total / 1024**2:.2f} MiB")


### 3.3 矩阵分块与任务划分

为使矩阵数据能够在片上统一缓冲区（Unified Buffer，UB）中完成计算，核函数沿$M$方向划分行块，沿$N$方向划分列块，并在每个输出块内部遍历$K$维度。

| 实现 | 行块大小 | 列块大小 | $K$最大长度 | 32字节对齐要求 |
|---|---:|---:|---:|---:|
| FP32 | 8行 | 1024列 | 4096 | $N$、$K$为8的倍数 |
| FP16 | 16行 | 1024列 | 4096 | $N$、$K$为16的倍数 |

设行块大小为$M_{tile}$，则行块数量为：

$$
numRowTiles=\left\lceil\frac{M}{M_{tile}}\right\rceil
$$

Host侧根据目标昇腾处理器可用的AI Vector核数量和行块数量确定核函数启动任务数。第`blockIndex`个逻辑任务按照`blockIndex`、`blockIndex + blockCount`、`blockIndex + 2 * blockCount`的顺序处理行块。不同逻辑任务负责互不重叠的输出行，不需要进行核间归约。

当前实现一次将一个$A$行块完整搬入UB，因此要求$K\le4096$。FP32和FP16的连续搬运长度分别按照8个元素和16个元素对齐，以满足32字节数据搬运要求。


### 3.4 Vector计算方法

Device侧使用`GlobalTensor`描述全局内存（Global Memory，GM）中的矩阵$A$、$B$和$C$，并在UB中申请固定容量的`LocalTensor`。各局部张量的作用如下：

- `aLocal`：保存当前$A$行块，并在多个列块之间复用；
- `bLocal`：保存矩阵$B$在当前$K$位置上的连续列数据；
- `aBroadcastLocal`：保存由一个$A$元素广播得到的向量；
- `productLocal`：保存`Mul`接口得到的乘法结果；
- `cLocal`：保存当前输出块的累加结果。

一个输出块的计算过程如下：

1. 使用`DataCopy`将$A$行块从GM搬入`aLocal`；
2. 将`cLocal`初始化为零；
3. 依次遍历$K$维度，将$B$的连续列数据搬入`bLocal`，并将对应的$A$元素广播至`aBroadcastLocal`；
4. 使用`Mul`计算乘积并写入`productLocal`，再使用`Add`将乘积累加到`cLocal`；
5. 完成$K$维度归约后，将`cLocal`中的结果写回GM。

核函数采用静态Tensor编程方式，不使用`TPipe`和`TQue`。MTE2、Scalar、Vector和MTE3流水线之间的数据依赖通过`SetFlag`和`WaitFlag`管理，确保局部张量在数据搬入、计算和写回完成之前不被覆盖。


### 3.5 Host侧与Device侧协同执行

Host侧负责实验数据准备、运行时管理和结果分析，Device侧负责低精度GEMM计算。完整执行流程如下：

1. Host侧生成FP64输入矩阵，并分别转换为FP32和FP16；
2. CPU使用FP64数据执行串行GEMM，生成参考结果并记录基线时间；
3. Host侧根据矩阵规模、数据精度和目标处理器信息生成任务配置，申请Device侧内存并完成输入数据搬运；
4. Device侧分别执行FP32或FP16 GEMM核函数，预热5次后重复运行30次；
5. Host侧将NPU结果回拷，并与CPU FP64参考结果逐元素比较；
6. 程序输出端到端总时间、加速比、最大绝对误差和最大相对误差。

其中，NPU端到端总时间包括输入数据从Host侧搬运至Device侧的时间、核函数平均执行时间和结果回拷时间。CPU FP64基线时间用于计算加速比，不单独作为本实验的结果指标。


### 3.6 实验参数设置

本实验的矩阵规模为$4096\times4096$，即$M=N=K=4096$；随机种子为`20260717`；NPU核函数预热5次、重复运行30次。运行Notebook时直接使用以下正式实验参数。


In [ ]:
M = N = K = 4096
WARMUP = 5
REPEAT = 30

SEED = 20260717
RELATIVE_DENOMINATOR_FLOOR = 1.0e-12

assert M > 0 and N > 0 and K > 0
assert K <= 4096, "当前核函数要求K不超过4096"
assert N % 16 == 0 and K % 16 == 0, "同时运行FP32和FP16时，N和K应为16的倍数"

print(
    {
        "shape": (M, N, K),
        "warmup": WARMUP,
        "repeat": REPEAT,
        "seed": SEED,
    }
)


---
## 4. 核函数开发

本节实现FP32和FP16两种精度的GEMM核函数。两种核函数采用相同的矩阵分块和Vector计算流程，并通过C++函数模板复用公共计算代码。FP32与FP16实现的主要差异是数据类型、行块大小和数据搬运对齐长度。

本实验采用静态Tensor编程方式，在UB中申请固定容量的`LocalTensor`，并使用`DataCopy`、`Duplicate`、`Mul`和`Add`等基础API完成数据搬运与矩阵计算。


### 4.1 GEMM分块计算公共实现

`RunStaticVectorGemm<T, RowTile>`是FP32与FP16核函数共用的分块计算函数。模板参数`T`表示计算数据类型，`RowTile`表示一个行块包含的矩阵行数。

该函数的主要流程如下：

1. 调用`InitSocState`初始化静态Tensor编程所需的处理器状态；
2. 使用`GlobalTensor`描述全局内存中的输入矩阵和输出矩阵；
3. 在UB中申请保存$A$行块、$B$列数据、广播数据、乘法结果和输出块的`LocalTensor`；
4. 根据`GetBlockIdx`和`GetBlockNum`确定当前逻辑任务负责的行块；
5. 将$A$行块搬入UB，并在多个列块计算中复用；
6. 遍历$K$维度，依次完成$B$数据搬入、$A$元素广播、向量乘法和结果累加；
7. 完成当前输出块计算后，将结果写回全局内存；
8. 在复用局部张量或退出核函数前，等待相关流水线事件执行完成。

`Mul`的输出先写入`productLocal`，再由`Add`累加至`cLocal`。因此，FP32实现使用FP32乘法结果和FP32累加结果，FP16实现使用FP16乘法结果和FP16累加结果。


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/op_kernel/vector_gemm_static.h
#ifndef LOW_PRECISION_GEMM_VECTOR_GEMM_STATIC_H
#define LOW_PRECISION_GEMM_VECTOR_GEMM_STATIC_H

#include "kernel_operator.h"

namespace lowp_device {

// The experiment fixes K at 4096 by default. Keeping one complete A row block
// in UB lets every N tile reuse it instead of moving the same A data again.
constexpr uint32_t kMaximumReduction = 4096;
constexpr uint32_t kColumnTile = 1024;

constexpr int32_t kAEvent = 0;
constexpr int32_t kBEvent = 1;
constexpr int32_t kCEvent = 2;

__aicore__ inline uint32_t Minimum(uint32_t lhs, uint32_t rhs)
{
    return lhs < rhs ? lhs : rhs;
}

template <typename T, uint32_t RowTile>
__aicore__ inline void RunStaticVectorGemm(
    GM_ADDR a,
    GM_ADDR b,
    GM_ADDR c,
    uint32_t m,
    uint32_t n,
    uint32_t k)
{
    // Static Tensor kernels do not construct TPipe, so the kernel must
    // initialize the SoC global state registers explicitly.
    AscendC::InitSocState();

    constexpr uint32_t aLocalElements = RowTile * kMaximumReduction;
    constexpr uint32_t cLocalElements = RowTile * kColumnTile;

    AscendC::GlobalTensor<T> aGlobal;
    AscendC::GlobalTensor<T> bGlobal;
    AscendC::GlobalTensor<T> cGlobal;
    aGlobal.SetGlobalBuffer(
        reinterpret_cast<__gm__ T *>(a),
        static_cast<uint64_t>(m) * k);
    bGlobal.SetGlobalBuffer(
        reinterpret_cast<__gm__ T *>(b),
        static_cast<uint64_t>(k) * n);
    cGlobal.SetGlobalBuffer(
        reinterpret_cast<__gm__ T *>(c),
        static_cast<uint64_t>(m) * n);

    // Static Tensor programming: UB addresses are assigned once, without
    // constructing TPipe/TQue. All tensors in this kernel have type T.
    AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    AscendC::LocalTensor<T> aLocal =
        ubAllocator.Alloc<AscendC::TPosition::VECCALC, T, aLocalElements>();
    AscendC::LocalTensor<T> bLocal =
        ubAllocator.Alloc<AscendC::TPosition::VECCALC, T, kColumnTile>();
    AscendC::LocalTensor<T> aBroadcastLocal =
        ubAllocator.Alloc<AscendC::TPosition::VECCALC, T, kColumnTile>();
    AscendC::LocalTensor<T> productLocal =
        ubAllocator.Alloc<AscendC::TPosition::VECCALC, T, kColumnTile>();
    AscendC::LocalTensor<T> cLocal =
        ubAllocator.Alloc<AscendC::TPosition::VECCALC, T, cLocalElements>();

    const uint32_t rowTileCount = (m + RowTile - 1U) / RowTile;
    // CANN 9.0 exposes these core-index helpers in the AscendC namespace.
    const uint32_t blockIndex = static_cast<uint32_t>(AscendC::GetBlockIdx());
    const uint32_t blockCount = static_cast<uint32_t>(AscendC::GetBlockNum());

    bool aReadPending = false;
    bool bReadPending = false;
    bool cWritePending = false;

    for (uint32_t rowTileIndex = blockIndex;
         rowTileIndex < rowTileCount;
         rowTileIndex += blockCount) {
        const uint32_t firstRow = rowTileIndex * RowTile;
        const uint32_t rowCount = Minimum(RowTile, m - firstRow);

        // The previous tile must finish reading aLocal before MTE2 overwrites it.
        if (aReadPending) {
            AscendC::WaitFlag<AscendC::HardEvent::S_MTE2>(kAEvent);
            aReadPending = false;
        }

        for (uint32_t localRow = 0; localRow < rowCount; ++localRow) {
            const uint64_t globalOffset =
                static_cast<uint64_t>(firstRow + localRow) * k;
            AscendC::DataCopy(
                aLocal[localRow * kMaximumReduction],
                aGlobal[globalOffset],
                k);
        }
        // GetValue below reads aLocal on the Scalar pipeline.
        AscendC::SetFlag<AscendC::HardEvent::MTE2_S>(kAEvent);
        AscendC::WaitFlag<AscendC::HardEvent::MTE2_S>(kAEvent);

        for (uint32_t firstColumn = 0; firstColumn < n; firstColumn += kColumnTile) {
            const uint32_t columnCount = Minimum(kColumnTile, n - firstColumn);

            // The previous MTE3 write must finish before cLocal is initialized
            // for the next output tile.
            if (cWritePending) {
                AscendC::WaitFlag<AscendC::HardEvent::MTE3_V>(kCEvent);
                cWritePending = false;
            }
            AscendC::Duplicate(
                cLocal,
                static_cast<T>(0),
                rowCount * kColumnTile);

            for (uint32_t reduction = 0; reduction < k; ++reduction) {
                // Do not let the next B row overwrite bLocal while Vector is
                // still consuming the current row.
                if (bReadPending) {
                    AscendC::WaitFlag<AscendC::HardEvent::V_MTE2>(kBEvent);
                    bReadPending = false;
                }

                const uint64_t bOffset =
                    static_cast<uint64_t>(reduction) * n + firstColumn;
                AscendC::DataCopy(
                    bLocal,
                    bGlobal[bOffset],
                    columnCount);
                AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(kBEvent);
                AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(kBEvent);

                for (uint32_t localRow = 0; localRow < rowCount; ++localRow) {
                    const T aValue =
                        aLocal.GetValue(localRow * kMaximumReduction + reduction);

                    // Deliberately keep multiply and accumulation separate.
                    // productLocal materializes and rounds the Mul result in T
                    // before Add updates the T-typed partial sum in cLocal.
                    AscendC::Duplicate(aBroadcastLocal, aValue, columnCount);
                    AscendC::Mul(
                        productLocal,
                        aBroadcastLocal,
                        bLocal,
                        columnCount);
                    AscendC::Add(
                        cLocal[localRow * kColumnTile],
                        cLocal[localRow * kColumnTile],
                        productLocal,
                        columnCount);
                }

                AscendC::SetFlag<AscendC::HardEvent::V_MTE2>(kBEvent);
                bReadPending = true;
            }

            AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(kCEvent);
            AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(kCEvent);
            for (uint32_t localRow = 0; localRow < rowCount; ++localRow) {
                const uint64_t cOffset =
                    static_cast<uint64_t>(firstRow + localRow) * n + firstColumn;
                AscendC::DataCopy(
                    cGlobal[cOffset],
                    cLocal[localRow * kColumnTile],
                    columnCount);
            }
            AscendC::SetFlag<AscendC::HardEvent::MTE3_V>(kCEvent);
            cWritePending = true;
        }

        // Pair every event before reusing local storage or leaving the kernel.
        if (bReadPending) {
            AscendC::WaitFlag<AscendC::HardEvent::V_MTE2>(kBEvent);
            bReadPending = false;
        }
        if (cWritePending) {
            AscendC::WaitFlag<AscendC::HardEvent::MTE3_V>(kCEvent);
            cWritePending = false;
        }
        AscendC::SetFlag<AscendC::HardEvent::S_MTE2>(kAEvent);
        aReadPending = true;
    }

    if (aReadPending) {
        AscendC::WaitFlag<AscendC::HardEvent::S_MTE2>(kAEvent);
    }
}

}  // namespace lowp_device

#endif  // LOW_PRECISION_GEMM_VECTOR_GEMM_STATIC_H


### 4.2 FP32 GEMM核函数

`fp32_vector_gemm`以`float`作为计算数据类型，并将行块大小设置为8。矩阵输入、广播数据、乘法结果、累加结果和输出结果均为FP32。`Mul`和`Add`分别完成向量乘法与向量加法，不使用Matmul高阶API或Cube计算单元。

FP32元素占4字节，因此矩阵$B$的连续列长度以及$K$维度长度需要按8个元素对齐，以满足32字节数据搬运要求。


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/op_kernel/vector_gemm_fp32.cpp
/**
 * Strict FP32 GEMM using Ascend C static Tensor programming and Vector basic APIs.
 *
 * Mul writes an FP32 product tensor first; Add then accumulates that FP32
 * tensor into an FP32 output tile. Matmul, Cube and MulAddDst are not used.
 */
#include "vector_gemm_static.h"

extern "C" __global__ __aicore__ void fp32_vector_gemm(
    GM_ADDR a,
    GM_ADDR b,
    GM_ADDR c,
    uint32_t m,
    uint32_t n,
    uint32_t k)
{
    constexpr uint32_t kFp32RowTile = 8;
    lowp_device::RunStaticVectorGemm<float, kFp32RowTile>(a, b, c, m, n, k);
}


### 4.3 FP16 GEMM核函数

`fp16_vector_gemm`以`half`作为计算数据类型，并将行块大小设置为16。矩阵输入、广播数据、乘法结果、累加结果和输出结果均为FP16。乘法结果写入`productLocal`时按照FP16精度保存，随后继续以FP16精度累加，不使用FP32中间结果。

FP16元素占2字节，因此矩阵$B$的连续列长度以及$K$维度长度需要按16个元素对齐。FP16行块包含的矩阵行数多于FP32，可以在相近的UB占用量下提高单个逻辑任务的处理粒度。


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/ascend_ops/op_kernel/vector_gemm_fp16.cpp
/**
 * Strict FP16 GEMM using Ascend C static Tensor programming and Vector basic APIs.
 *
 * Every input, broadcast tensor, product tensor, partial sum and output is
 * half. Mul and Add are separate instructions, so no FP32 multiply or FP32
 * accumulation path is present in this kernel.
 */
#include "vector_gemm_static.h"

extern "C" __global__ __aicore__ void fp16_vector_gemm(
    GM_ADDR a,
    GM_ADDR b,
    GM_ADDR c,
    uint32_t m,
    uint32_t n,
    uint32_t k)
{
    constexpr uint32_t kFp16RowTile = 16;
    lowp_device::RunStaticVectorGemm<half, kFp16RowTile>(a, b, c, m, n, k);
}


### 4.4 核函数实现要点

1. **数据复用**：一个$A$行块只搬入UB一次，并在多个$N$方向列块中复用，从而减少对矩阵$A$的重复访问。
2. **连续搬运**：对固定的$K$维度位置，矩阵$B$按连续列区间搬入UB，便于使用`DataCopy`和Vector基础API处理。
3. **精度一致性**：FP32和FP16核函数分别使用对应精度的输入、乘法结果、累加结果和输出，确保实验能够直接比较不同计算精度产生的误差。
4. **任务独立性**：不同逻辑任务负责互不重叠的输出行块，不存在多个任务同时写入同一输出元素的问题。
5. **流水线同步**：数据搬入、Scalar读取、Vector计算和结果写回之间通过事件同步，局部张量只能在前一次使用结束后复用。
6. **边界约束**：当前实现要求$K\le4096$，并要求$N$和$K$满足对应数据类型的32字节对齐条件。若需要支持更大的$K$，应增加$K$方向分块及分块结果累加过程。


In [ ]:
kernel_files = [
    WORK_DIR / "ascend_ops" / "op_kernel" / "vector_gemm_static.h",
    WORK_DIR / "ascend_ops" / "op_kernel" / "vector_gemm_fp32.cpp",
    WORK_DIR / "ascend_ops" / "op_kernel" / "vector_gemm_fp16.cpp",
]

for path in kernel_files:
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    print(
        f"{'OK' if text else 'MISSING':7s}",
        f"{len(text.splitlines()):4d} lines",
        path.relative_to(WORK_DIR),
    )

static_source = kernel_files[0].read_text(encoding="utf-8")
for token in ["LocalMemAllocator", "DataCopy", "Duplicate", "Mul(", "Add(", "SetFlag", "WaitFlag"]:
    assert token in static_source, f"公共计算实现中缺少{token}"
print("静态Tensor编程与Vector基础API检查：OK")


---
## 5. 结果验证与性能分析

核函数开发完成后，本节按照数据准备、工程构建、算子运行、结果验证和性能分析五个环节组织实验。运行本Notebook后，`src/04.01_extra_low_precision_gemm`目录具备从CPU基线到NPU FP32/FP16实验的完整工程文件。


### 5.1 数据准备

Host侧根据$M$、$N$、$K$和随机种子生成FP64输入。FP32与FP16程序各自从同一随机规则重新生成FP64输入并转换，因此两次实验的数据内容一致。CPU参考与NPU结果共用同一形状和行主序布局。

正式构建前再次检查目前已写入的源码。该检查不依赖CANN，可在普通Jupyter环境中完成。


In [ ]:
source_files = public_files + kernel_files
missing = [path for path in source_files if not path.exists() or path.stat().st_size == 0]
print(f"source files: {len(source_files)}, missing: {len(missing)}")
for path in missing:
    print("MISSING", path)
assert not missing


### 5.2 工程构建

`CMakeLists.txt`始终构建CPU FP64程序；设置`BUILD_ASCEND=ON`后，再引入CANN的`ascendc.cmake`，编译FP32/FP16核函数库与两个NPU Host程序，并链接ACL运行库。

CPU基线在没有CANN的普通C++17环境中也可构建。NPU构建需要有效的CANN安装路径和目标`SOC_VERSION`。


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(low_precision_gemm LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
    set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build the Ascend C static-Tensor Vector GEMM experiments" OFF)

# The CPU FP64 implementation is always available. This keeps the correctness
# baseline runnable on an ordinary compiler without requiring CANN.
add_library(gemm_cpu_support STATIC
    src/gemm_common.cpp
    src/gemm_fp64_cpu.cpp
)
target_include_directories(gemm_cpu_support PUBLIC
    "${CMAKE_CURRENT_SOURCE_DIR}/include"
)
target_compile_options(gemm_cpu_support PRIVATE
    -O3
    -Wall
    -Wextra
    -Wpedantic
)

add_executable(gemm_fp64_cpu_demo
    src/main_fp64_cpu.cpp
)
target_link_libraries(gemm_fp64_cpu_demo PRIVATE gemm_cpu_support)
target_compile_options(gemm_fp64_cpu_demo PRIVATE
    -O3
    -Wall
    -Wextra
    -Wpedantic
)

if(BUILD_ASCEND)
    # CANN host-side targets use libstdc++'s legacy ABI. Apply the same ABI to
    # the shared CPU support library and propagate it to all consumers. This
    # keeps std::string symbols such as HumanReadableBytes consistent at link
    # time while leaving ordinary BUILD_ASCEND=OFF builds on the system ABI.
    target_compile_definitions(gemm_cpu_support PUBLIC
        _GLIBCXX_USE_CXX11_ABI=0
    )

    set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu or sim")
    set(SOC_VERSION "Ascend910B2" CACHE STRING "Ascend SoC compiler target")
    set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}"
        CACHE PATH "CANN toolkit installation directory")
    if(NOT ASCEND_CANN_PATH)
        set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest"
            CACHE PATH "CANN toolkit installation directory" FORCE)
    endif()
    set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}"
        CACHE PATH "CANN package path used by Ascend C CMake" FORCE)
    set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out"
        CACHE PATH "Ascend C generated headers and install output" FORCE)

    if(NOT RUN_MODE STREQUAL "npu" AND NOT RUN_MODE STREQUAL "sim")
        message(FATAL_ERROR "RUN_MODE must be npu or sim")
    endif()

    if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
        set(ASCENDC_CMAKE_FILE
            "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
        set(ASCENDC_CMAKE_FILE
            "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
        set(ASCENDC_CMAKE_FILE
            "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    else()
        message(FATAL_ERROR
            "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. "
            "Set ASCEND_INSTALL_PATH or ASCEND_CANN_PATH to a valid CANN package.")
    endif()

    message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
    message(STATUS "SOC_VERSION=${SOC_VERSION}")
    message(STATUS "RUN_MODE=${RUN_MODE}")
    include("${ASCENDC_CMAKE_FILE}")

    ascendc_library(gemm_vector_kernels STATIC
        ascend_ops/op_kernel/vector_gemm_fp32.cpp
        ascend_ops/op_kernel/vector_gemm_fp16.cpp
    )

    add_executable(gemm_fp32_npu_demo
        ascend_ops/host_launch/main_fp32_npu.cpp
        ascend_ops/host_launch/vector_gemm_config.cpp
    )
    add_executable(gemm_fp16_npu_demo
        ascend_ops/host_launch/main_fp16_npu.cpp
        ascend_ops/host_launch/vector_gemm_config.cpp
    )

    foreach(ASCEND_DEMO_TARGET IN ITEMS gemm_fp32_npu_demo gemm_fp16_npu_demo)
        target_include_directories(${ASCEND_DEMO_TARGET} PRIVATE
            "${CMAKE_CURRENT_SOURCE_DIR}/include"
            "${CMAKE_CURRENT_SOURCE_DIR}/ascend_ops/host_launch"
            "${CMAKE_BINARY_DIR}/out/include/gemm_vector_kernels"
            "${CMAKE_INSTALL_PREFIX}/include/gemm_vector_kernels"
        )
        # Vendor headers intentionally use macro forms that trigger -Wpedantic.
        target_include_directories(${ASCEND_DEMO_TARGET} SYSTEM PRIVATE
            "${ASCEND_CANN_PACKAGE_PATH}/include"
            "${ASCEND_CANN_PACKAGE_PATH}/runtime/include"
        )
        target_link_directories(${ASCEND_DEMO_TARGET} PRIVATE
            "${ASCEND_CANN_PACKAGE_PATH}/lib64"
            "${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64"
            "${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub"
        )
        target_compile_definitions(${ASCEND_DEMO_TARGET} PRIVATE
            SOC_VERSION="${SOC_VERSION}"
        )
        target_compile_options(${ASCEND_DEMO_TARGET} PRIVATE
            -O3
            -Wall
            -Wextra
            -Wpedantic
        )
        target_link_libraries(${ASCEND_DEMO_TARGET} PRIVATE
            gemm_vector_kernels
            gemm_cpu_support
            host_intf_pub
            ascendcl
            platform
            ascendalog
            dl
        )
        add_dependencies(${ASCEND_DEMO_TARGET} gemm_vector_kernels)
    endforeach()
endif()

install(TARGETS gemm_fp64_cpu_demo RUNTIME DESTINATION bin)
if(BUILD_ASCEND)
    install(
        TARGETS gemm_fp32_npu_demo gemm_fp16_npu_demo
        RUNTIME DESTINATION bin
    )
endif()


### 5.3 算子运行

下面写入CPU与NPU运行脚本。CPU脚本构建并运行串行FP64基线；NPU公共脚本负责检测CANN、目标SoC和Device，构建指定精度程序，并将实验参数传给Host程序。FP32与FP16包装脚本调用同一个公共脚本。

NPU程序使用4096规模、5次预热和30次重复，并分别生成`results/fp32_result.json`与`results/fp16_result.json`。Notebook运行单元直接使用第3.6节设置的正式实验参数。


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/scripts/run_cpu_fp64.sh
#!/usr/bin/env bash
set -euo pipefail

PROJECT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"
BUILD_DIR="${PROJECT_DIR}/build_cpu_fp64"

cmake -S "${PROJECT_DIR}" -B "${BUILD_DIR}" \
    -DCMAKE_BUILD_TYPE=Release \
    -DBUILD_ASCEND=OFF
cmake --build "${BUILD_DIR}" --target gemm_fp64_cpu_demo -j "${BUILD_JOBS:-8}"

"${BUILD_DIR}/gemm_fp64_cpu_demo" "$@"


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/scripts/run_ascend_vector.sh
#!/usr/bin/env bash
set -euo pipefail

if [[ $# -lt 1 || ( "$1" != "fp32" && "$1" != "fp16" ) ]]; then
    echo "Usage: bash scripts/run_ascend_vector.sh {fp32|fp16} [options]" >&2
    exit 2
fi

PRECISION="$1"
shift

PROJECT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"
BUILD_DIR="${PROJECT_DIR}/build_ascend_vector_${PRECISION}"
RESULT_DIR="${PROJECT_DIR}/results"
TARGET="gemm_${PRECISION}_npu_demo"

ASCEND_PATH=""
SOC_NAME="${SOC_VERSION:-}"
DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=0
BUILD_ONLY=0
PROGRAM_ARGS=()

is_cann_path() {
    local path="$1"
    [[ -f "${path}/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
       -f "${path}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
       -f "${path}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ]]
}

detect_cann_path() {
    local architecture candidate root
    architecture="$(uname -m)"

    for root in \
        "${ASCEND_INSTALL_PATH:-}" \
        "${ASCEND_HOME_PATH:-}" \
        "${ASCEND_TOOLKIT_HOME:-}"; do
        [[ -n "${root}" ]] || continue
        for candidate in "${root}" "${root}/${architecture}-linux"; do
            if is_cann_path "${candidate}"; then
                echo "${candidate}"
                return 0
            fi
        done
    done

    for candidate in \
        /opt/conda/Ascend/cann-*/"${architecture}-linux" \
        /usr/local/Ascend/ascend-toolkit/latest \
        /usr/local/Ascend/ascend-toolkit/*/"${architecture}-linux"; do
        if is_cann_path "${candidate}"; then
            echo "${candidate}"
            return 0
        fi
    done
    return 1
}

canonicalize_soc_version() {
    local raw_name="$1"
    local normalized chip_name
    normalized="$(
        echo "${raw_name}" |
            tr '[:upper:]' '[:lower:]' |
            tr -cd '[:alnum:]'
    )"
    case "${normalized}" in
        ascend910*|ascend310*) chip_name="${normalized#ascend}" ;;
        910*|310*) chip_name="${normalized}" ;;
        *) return 1 ;;
    esac
    chip_name="$(echo "${chip_name}" | tr '[:lower:]' '[:upper:]')"
    echo "Ascend${chip_name}"
}

detect_soc_version() {
    local raw_name
    command -v npu-smi >/dev/null 2>&1 || return 1
    raw_name="$(
        npu-smi info 2>/dev/null |
            sed -nE 's/^\|[[:space:]]*[0-9]+[[:space:]]+([^|]+)\|.*/\1/p' |
            head -n 1 || true
    )"
    canonicalize_soc_version "${raw_name}"
}

usage() {
    cat <<USAGE
Usage: bash scripts/run_ascend_${PRECISION}.sh [build options] [-- program options]

Build options:
  -a, --install-path PATH    CANN path; auto-detected when omitted
  -v, --soc-version SOC      e.g. Ascend910B4; auto-detected when omitted
  -d, --device ID            ACL logical device id, default: 0
  -r, --run-mode MODE        npu or sim, default: npu
  -t, --build-type TYPE      CMake build type, default: Release
  -c, --clean                remove the shared Ascend Vector build directory
      --build-only           compile without running
  -h, --help                 show this message

Program options after "--" are passed to ${TARGET}.

Examples:
  bash scripts/run_ascend_${PRECISION}.sh -v Ascend910B4
  bash scripts/run_ascend_${PRECISION}.sh -- --m 256 --n 256 --k 256 --repeat 1
USAGE
}

require_value() {
    if [[ $# -lt 2 ]]; then
        echo "[ERROR] missing value after $1" >&2
        exit 2
    fi
}

while [[ $# -gt 0 ]]; do
    case "$1" in
        -a|--install-path)
            require_value "$@"
            ASCEND_PATH="$2"
            shift 2
            ;;
        -v|--soc-version)
            require_value "$@"
            SOC_NAME="$2"
            shift 2
            ;;
        -d|--device)
            require_value "$@"
            DEVICE_ID="$2"
            shift 2
            ;;
        -r|--run-mode)
            require_value "$@"
            RUN_MODE="$2"
            shift 2
            ;;
        -t|--build-type)
            require_value "$@"
            BUILD_TYPE="$2"
            shift 2
            ;;
        -c|--clean)
            CLEAN=1
            shift
            ;;
        --build-only)
            BUILD_ONLY=1
            shift
            ;;
        -h|--help)
            usage
            exit 0
            ;;
        --)
            shift
            PROGRAM_ARGS=("$@")
            break
            ;;
        *)
            echo "[ERROR] unknown build option: $1" >&2
            usage
            exit 2
            ;;
    esac
done

if [[ "${RUN_MODE}" != "npu" && "${RUN_MODE}" != "sim" ]]; then
    echo "[ERROR] --run-mode must be npu or sim" >&2
    exit 2
fi
if [[ ! "${DEVICE_ID}" =~ ^[0-9]+$ ]]; then
    echo "[ERROR] --device must be a non-negative integer" >&2
    exit 2
fi

if [[ -z "${ASCEND_PATH}" ]]; then
    if ! ASCEND_PATH="$(detect_cann_path)"; then
        echo "[ERROR] cannot find CANN; pass --install-path PATH" >&2
        exit 1
    fi
elif ! is_cann_path "${ASCEND_PATH}"; then
    echo "[ERROR] invalid CANN path: ${ASCEND_PATH}" >&2
    exit 1
fi

if [[ -z "${SOC_NAME}" ]]; then
    if ! SOC_NAME="$(detect_soc_version)"; then
        echo "[ERROR] cannot detect SOC_VERSION from npu-smi; pass --soc-version SOC" >&2
        exit 1
    fi
elif ! SOC_NAME="$(canonicalize_soc_version "${SOC_NAME}")"; then
    echo "[ERROR] unsupported SOC_VERSION: ${SOC_NAME}" >&2
    echo "[ERROR] expected a value such as Ascend910B4 or Ascend310P3" >&2
    exit 2
fi

if [[ "${CLEAN}" -eq 1 ]]; then
    rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}" "${RESULT_DIR}"

if [[ -f "${ASCEND_PATH}/set_env.sh" ]]; then
    # shellcheck disable=SC1090
    source "${ASCEND_PATH}/set_env.sh"
elif [[ -f "${ASCEND_PATH}/bin/setenv.bash" ]]; then
    # shellcheck disable=SC1090
    source "${ASCEND_PATH}/bin/setenv.bash"
fi

# CANN 9.0 notebook packages may expose tikcpp/ and ccec_compiler/ at the
# package root while ascendc.cmake still expects them below ascendc_devkit/.
ASCEND_CMAKE_PATH="${ASCEND_PATH}"
ASCENDC_NEW_LAYOUT="${ASCEND_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
BISHENG_REAL="${ASCEND_PATH}/ccec_compiler/bin/bisheng"
BISHENG_EXPECTED="${ASCEND_PATH}/ascendc_devkit/ccec_compiler/bin/bisheng"
if [[ -f "${ASCENDC_NEW_LAYOUT}" && -x "${BISHENG_REAL}" && ! -x "${BISHENG_EXPECTED}" ]]; then
    ASCEND_CMAKE_PATH="${BUILD_DIR}/cann_package_compat"
    mkdir -p "${ASCEND_CMAKE_PATH}/ascendc_devkit"

    for entry in "${ASCEND_PATH}"/*; do
        [[ -e "${entry}" ]] || continue
        name="$(basename "${entry}")"
        [[ "${name}" == "ascendc_devkit" ]] && continue
        ln -sfn "${entry}" "${ASCEND_CMAKE_PATH}/${name}"
    done

    if [[ -d "${ASCEND_PATH}/ascendc_devkit" ]]; then
        for entry in "${ASCEND_PATH}/ascendc_devkit"/*; do
            [[ -e "${entry}" ]] || continue
            name="$(basename "${entry}")"
            [[ "${name}" == "ccec_compiler" ]] && continue
            ln -sfn "${entry}" "${ASCEND_CMAKE_PATH}/ascendc_devkit/${name}"
        done
    fi

    for component in asc tikcpp ccec_compiler; do
        if [[ -e "${ASCEND_PATH}/${component}" ]]; then
            ln -sfn "${ASCEND_PATH}/${component}" \
                "${ASCEND_CMAKE_PATH}/ascendc_devkit/${component}"
        fi
    done
    echo "[INFO] using CANN compatibility view: ${ASCEND_CMAKE_PATH}"
fi

export ASCEND_INSTALL_PATH="${ASCEND_PATH}"
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PATH}"
export SOC_VERSION="${SOC_NAME}"

echo "[INFO] CANN=${ASCEND_PATH}"
echo "[INFO] PRECISION=${PRECISION}, SOC_VERSION=${SOC_NAME}, RUN_MODE=${RUN_MODE}, DEVICE_ID=${DEVICE_ID}"

cmake -S "${PROJECT_DIR}" -B "${BUILD_DIR}" \
    -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
    -DBUILD_ASCEND=ON \
    -DASCEND_CANN_PATH="${ASCEND_CMAKE_PATH}" \
    -DASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PATH}" \
    -DSOC_VERSION="${SOC_NAME}" \
    -DRUN_MODE="${RUN_MODE}"
cmake --build "${BUILD_DIR}" --target "${TARGET}" -j "${BUILD_JOBS:-8}"

EXECUTABLE="${BUILD_DIR}/${TARGET}"
if [[ ! -x "${EXECUTABLE}" ]]; then
    echo "[ERROR] executable was not generated: ${EXECUTABLE}" >&2
    exit 1
fi
if [[ "${BUILD_ONLY}" -eq 1 ]]; then
    echo "[INFO] build complete: ${EXECUTABLE}"
    exit 0
fi

if [[ ${#PROGRAM_ARGS[@]} -eq 0 ]]; then
    PROGRAM_ARGS=(
        --m 4096
        --n 4096
        --k 4096
        --warmup 5
        --repeat 30
        --json "${RESULT_DIR}/${PRECISION}_result.json"
    )
fi

export LD_LIBRARY_PATH="${ASCEND_PATH}/lib64:${ASCEND_PATH}/runtime/lib64:${LD_LIBRARY_PATH:-}"
COMMAND=("${EXECUTABLE}" --device "${DEVICE_ID}")
COMMAND+=("${PROGRAM_ARGS[@]}")
echo "[RUN] ${COMMAND[*]}"
"${COMMAND[@]}"


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/scripts/run_ascend_fp32.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
exec bash "${SCRIPT_DIR}/run_ascend_vector.sh" fp32 "$@"


In [ ]:
%%writefile src/04.01_extra_low_precision_gemm/scripts/run_ascend_fp16.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
exec bash "${SCRIPT_DIR}/run_ascend_vector.sh" fp16 "$@"


In [ ]:
for path in (WORK_DIR / "scripts").glob("*.sh"):
    path.chmod(path.stat().st_mode | 0o111)
    print("executable:", path.relative_to(WORK_DIR))


#### 5.3.1 运行CPU FP64版本

CPU程序验证输入生成、FP64参考实现和计时流程。CPU计时是NPU程序中加速比的参考，但不同机器上的CPU时间不可直接横向比较。

如果当前环境没有CMake，下面的单元会尝试使用可用的C++编译器直接构建同一组源码；两种构建路径的程序逻辑一致。


In [ ]:
cpu_result_path = WORK_DIR / "results" / "cpu_fp64_result.txt"
cpu_args = [
    "--m", str(M),
    "--n", str(N),
    "--k", str(K),
    "--seed", str(SEED),
    "--warmup", "1",
    "--repeat", "3",
    "--print-output",
]

cpu_command = None
if CMAKE_AVAILABLE:
    cpu_command = ["bash", "scripts/run_cpu_fp64.sh", *cpu_args]
else:
    compiler = shutil.which("c++") or shutil.which("g++") or shutil.which("clang++")
    if compiler:
        fallback_build = WORK_DIR / "build_cpu_fallback"
        fallback_build.mkdir(parents=True, exist_ok=True)
        executable = fallback_build / "gemm_fp64_cpu_demo"
        compile_command = [
            compiler, "-std=c++17", "-O3",
            "-I", str(WORK_DIR / "include"),
            str(WORK_DIR / "src" / "gemm_common.cpp"),
            str(WORK_DIR / "src" / "gemm_fp64_cpu.cpp"),
            str(WORK_DIR / "src" / "main_fp64_cpu.cpp"),
            "-o", str(executable),
        ]
        print("CMake unavailable; direct compile:", " ".join(compile_command))
        subprocess.run(compile_command, check=True)
        cpu_command = [str(executable), *cpu_args]

if cpu_command is None:
    print("No CMake or C++ compiler found; CPU run skipped.")
else:
    completed = subprocess.run(
        cpu_command,
        cwd=WORK_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )
    print(completed.stdout)
    cpu_result_path.write_text(completed.stdout, encoding="utf-8")
    print("CPU log written to:", cpu_result_path)


#### 5.3.2 运行NPU FP32与FP16版本

在已安装CANN且存在可用NPU的ModelArts环境中，下面的单元依次运行FP32和FP16程序。每个程序内部都会生成同一份FP64输入、计算CPU FP64参考、执行目标精度的NPU核函数并写出JSON。

运行条件不满足时，单元会安全跳过，不影响后续阅读和源码检查。若自动识别SoC失败，可在命令中为脚本增加`--soc-version Ascend910B4`等与实际设备一致的参数。


In [ ]:
if CANN_AVAILABLE and NPU_AVAILABLE and CMAKE_AVAILABLE:
    for precision in ["fp32", "fp16"]:
        result_path = WORK_DIR / "results" / f"{precision}_result.json"
        command = [
            "bash",
            f"scripts/run_ascend_{precision}.sh",
            "--",
            "--m", str(M),
            "--n", str(N),
            "--k", str(K),
            "--seed", str(SEED),
            "--warmup", str(WARMUP),
            "--repeat", str(REPEAT),
            "--relative-floor", str(RELATIVE_DENOMINATOR_FLOOR),
            "--json", str(result_path),
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, cwd=WORK_DIR, check=True)
else:
    missing_items = []
    if not CANN_AVAILABLE:
        missing_items.append("CANN")
    if not NPU_AVAILABLE:
        missing_items.append("NPU")
    if not CMAKE_AVAILABLE:
        missing_items.append("CMake")
    print("NPU experiments skipped; unavailable:", ", ".join(missing_items))


### 5.4 结果验证

本实验仅使用最大绝对误差和最大相对误差两个指标验证计算结果。设CPU FP64参考结果为$C^{ref}$，NPU低精度计算结果为$C^{npu}$，则最大绝对误差为：

$$
max\_abs=\max_i\left|C_i^{npu}-C_i^{ref}\right|
$$

最大相对误差为：

$$
max\_rel=\max_i
\frac{\left|C_i^{npu}-C_i^{ref}\right|}
{\max\left(\left|C_i^{ref}\right|,10^{-12}\right)}
$$

最大绝对误差反映结果偏差的最大幅度；最大相对误差反映误差相对于参考值大小的比例。当参考值接近零时，相对误差可能被放大，因此分析时应结合两个指标共同判断。

进行正确性验证时，应确保CPU与NPU使用相同的矩阵规模、随机种子和数据存储顺序。一般情况下，FP16的误差会大于FP32；若误差明显异常，应依次检查Host侧数据类型转换、矩阵偏移计算、$K$维度累加范围以及结果回拷过程。


In [ ]:
result_rows = []
for precision in ["fp32", "fp16"]:
    path = WORK_DIR / "results" / f"{precision}_result.json"
    print(f"===== {precision.upper()}：{path} =====")
    if not path.exists():
        print("未找到结果文件，请在CANN和NPU环境中运行对应实验。")
        continue

    data = json.loads(path.read_text(encoding="utf-8"))
    if "metrics" in data:
        metrics = data["metrics"]
        total_ms = metrics["end_to_end_total_ms"]
        speedup_value = metrics["speedup"]
        max_abs = metrics["max_absolute_error"]
        max_rel = metrics["max_relative_error"]
    else:
        # 兼容修改前生成的结果文件，但仅提取本实验规定的四项指标。
        timing = data["timing_ms"]
        error = data["error"]
        speedup_data = data["speedup"]
        total_ms = timing[f"npu_{precision}_end_to_end_one_run"]
        speedup_value = speedup_data["cpu_over_npu_end_to_end"]
        max_abs = error["max_absolute"]
        max_rel = error["max_relative"]

    result_rows.append(
        {
            "精度": precision.upper(),
            "端到端总时间/ms": total_ms,
            "加速比": speedup_value,
            "最大绝对误差": max_abs,
            "最大相对误差": max_rel,
        }
    )

if result_rows:
    columns = ["精度", "端到端总时间/ms", "加速比", "最大绝对误差", "最大相对误差"]
    print("\n===== FP32与FP16实验结果 =====")
    print(" | ".join(columns))
    print(" | ".join(["---"] * len(columns)))
    for row in result_rows:
        values = [
            row["精度"],
            f'{row["端到端总时间/ms"]:.6g}',
            f'{row["加速比"]:.6g}',
            f'{row["最大绝对误差"]:.6g}',
            f'{row["最大相对误差"]:.6g}',
        ]
        print(" | ".join(values))


### 5.5 性能分析

本实验仅使用端到端总时间和加速比两个指标分析执行性能。

NPU端到端总时间由输入数据搬运时间、核函数平均执行时间和结果回拷时间组成：

$$
T_{NPU}=T_{H2D}+T_{kernel}+T_{D2H}
$$

加速比定义为CPU FP64基线时间与NPU端到端总时间的比值：

$$
S=\frac{T_{CPU\_FP64}}{T_{NPU}}
$$

当$S>1$时，表示NPU低精度实现的端到端总时间小于CPU FP64基线时间；当$S<1$时，表示当前矩阵规模下数据搬运和核函数启动等开销尚未被并行计算收益抵消。

比较FP32与FP16时，应固定矩阵规模、随机种子、预热次数、重复次数、CANN版本和NPU设备。正式实验采用5次预热、30次重复运行，并以重复运行的核函数平均时间计算端到端总时间，以降低偶然波动对实验结果的影响。


---
## 6. 实验总结

本实验按照实验概述、环境准备、问题分析、核函数开发以及结果验证与性能分析五个阶段，实现了CPU FP64串行GEMM以及NPU FP32、FP16 GEMM。

* 问题分析明确了GEMM的输入输出、数据类型、连续存储方式、矩阵分块方法和Host侧与Device侧协同执行流程。
* Host侧从同一组FP64输入生成FP32和FP16数据，并以CPU FP64计算结果作为正确性验证基线。
* Device侧采用静态Tensor编程方式，在UB中申请固定容量的`LocalTensor`，使用Vector基础API完成数据搬运、乘法、累加和结果写回。
* FP32与FP16核函数采用相同的计算流程，并分别保持FP32和FP16的乘法及累加精度。
* 正确性验证使用最大绝对误差和最大相对误差，性能分析使用端到端总时间和加速比。

通过本实验，可以理解降低数据精度对矩阵存储、数据搬运、计算性能和数值误差的影响，掌握Ascend C静态Tensor编程中的矩阵分块、Vector计算和流水线同步方法，并能够从正确性与性能两个方面分析低精度GEMM的实验结果。
